# V2 MSc project — viva master code notebook

Student: Syed Safi Ullah · A00073183

Local viva notebook only. Do not commit. Do not upload to GitHub.

This file consolidates **actual V2 implementation** for a live code walkthrough.
It does not rerun the 420-case benchmark, calibration, judge, statistics, error analysis, or KB rebuild.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# git remote origin (discovered with `git remote -v`)
REPO_URL = "https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git"
REPO_NAME = "CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-"
REPO_DIR = Path("/content") / REPO_NAME


def _looks_like_repo(path: Path) -> bool:
    return (path / "V2" / "app" / "streamlit_app.py").is_file()


if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print("Repository already present — clone skipped.")
    os.chdir(REPO_DIR)
else:
    here = Path.cwd().resolve()
    local = None
    for cand in [here, here.parent, here.parent.parent]:
        if _looks_like_repo(cand):
            local = cand
            break
    if local is None and _looks_like_repo(Path('/Users/syedsafiullah/Documents/CAPSTONE (RAG WITH UNCERTAINITY QUANTIFICATION)')):
        local = Path('/Users/syedsafiullah/Documents/CAPSTONE (RAG WITH UNCERTAINITY QUANTIFICATION)')
    if local is None:
        raise RuntimeError("Not on Colab and no local V2 checkout was found.")
    REPO_DIR = local
    os.chdir(REPO_DIR)
    print("Local checkout — clone skipped.")

print("Repository:", REPO_DIR)
print("V2 exists:", (REPO_DIR / "V2").exists())


In [ ]:
from pathlib import Path
import os

V2_DIR = REPO_DIR / "V2"
os.chdir(V2_DIR)
if str(V2_DIR) not in sys.path:
    sys.path.insert(0, str(V2_DIR))
os.environ["PYTHONPATH"] = str(V2_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("V2 directory:", V2_DIR)
print("cwd:", Path.cwd())
needed = ["src", "app", "data", "results", "notebooks", "config", "scripts"]
for name in needed:
    p = V2_DIR / name
    print(("OK     " if p.exists() else "MISSING"), name)
print("Top-level V2:")
for p in sorted(V2_DIR.iterdir()):
    if not p.name.startswith("."):
        print(" ", p.name)


## GPU / T4 check

Environment only. Does not load Qwen or start a benchmark.

In [ ]:
# Hardware check only.
!nvidia-smi

import shutil
import subprocess
import sys

print("Python:", sys.version.replace("\n", " "))

gpu_name = None
vram = None
cuda_ok = False
cuda_ver = None
try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    print("CUDA available:", cuda_ok)
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        cuda_ver = torch.version.cuda
        props = torch.cuda.get_device_properties(0)
        vram = round(props.total_memory / (1024 ** 3), 2)
        print("GPU:", gpu_name)
        print("CUDA version:", cuda_ver)
        print("GPU VRAM (GB):", vram)
    else:
        print("GPU: none")
except Exception as exc:
    print("torch import/CUDA check failed:", exc)

print()
print("GPU CHECK")
print("---------")
print("Python:", sys.version.split()[0])
print("GPU available:", bool(cuda_ok))
print("GPU name:", gpu_name or "none")
print("CUDA available:", cuda_ok)
print("CUDA version:", cuda_ver)
print("GPU VRAM:", vram)
if cuda_ok and gpu_name and "tesla t4" in gpu_name.lower():
    print("Status: READY (Tesla T4)")
elif cuda_ok:
    print("WARNING: Expected Tesla T4 was not detected.")
    print("Detected GPU:", gpu_name)
else:
    print("Status: no CUDA GPU in this runtime")


In [ ]:
import importlib.util
import sys
from pathlib import Path

print("V2 VIVA ENVIRONMENT")
print("-------------------")
print("Python:", sys.version.split()[0])
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    print("CUDA:", torch.version.cuda if torch.cuda.is_available() else "not available")
except Exception as exc:
    print("GPU/CUDA: torch missing", type(exc).__name__)

def _mod(name):
    try:
        __import__(name)
        return "import OK"
    except Exception as exc:
        return f"missing ({type(exc).__name__})"

print("llama_cpp:", _mod("llama_cpp"))
print("chromadb:", _mod("chromadb"))
print("streamlit:", _mod("streamlit"))
print("sentence_transformers:", _mod("sentence_transformers"))

qwen = "not found on disk (this cell does not download weights)"
for root in [Path.cwd(), Path("/content"), Path.home() / ".cache" / "huggingface" / "hub"]:
    if not root.exists():
        continue
    try:
        found = list(root.rglob("*Qwen*Q4_K_M*.gguf"))
    except Exception:
        found = []
    if found:
        qwen = str(found[0])
        break
print("Qwen3 GGUF:", qwen)


## VIVA_MODE safety configuration

In [ ]:
# === VIVA-SAFE FLAGS (default = do not rerun the experiment) ===
VIVA_MODE = True
RUN_BENCHMARK = False
RUN_CALIBRATION = False
RUN_JUDGE = False
RUN_STATISTICS = False
RUN_ERROR_ANALYSIS = False
REBUILD_KB = False
RUN_LIVE_DEMO = True  # Phase 21 launches the existing Streamlit app only

print("VIVA MODE")
print("---------")
print("VIVA_MODE:", VIVA_MODE)
print("Benchmark:", "DISABLED" if not RUN_BENCHMARK else "ENABLED")
print("Calibration:", "DISABLED" if not RUN_CALIBRATION else "ENABLED")
print("Judge:", "DISABLED" if not RUN_JUDGE else "ENABLED")
print("Statistics:", "DISABLED" if not RUN_STATISTICS else "ENABLED")
print("Error analysis:", "DISABLED" if not RUN_ERROR_ANALYSIS else "ENABLED")
print("KB rebuild:", "DISABLED" if not REBUILD_KB else "ENABLED")
print("Live demo launch:", "ENABLED" if RUN_LIVE_DEMO else "DISABLED")
if VIVA_MODE:
    print("Expensive historical cells are labelled DO NOT RUN DURING VIVA and will skip.")


## 21-phase navigation

Names below are copied from `V2/project_record/PROJECT_MASTER_RECORD.md`.
Phases 1–6 and 17–20 have no Colab notebook; implementation lives in `V2/src/` and `V2/scripts/`.

| Phase | Actual phase name | Actual notebook / file | Main purpose | Important outputs | Master code IDs | Run during viva? |
|---|---|---|---|---|---|---|
| 1 | Project foundation | `V2/src/config/loader.py`, `V2/scripts/health_check.py` | V2 skeleton, YAML config, run IDs | `config/experiment.yaml` | M001–M003 | No |
| 2 | V1 audit + FinQA live profile | `V2/src/data/profile_finqa.py` | Live-load FinQA schema/splits | `data/processed/finqa_profile.json` | M004–M005 | No |
| 3 | Dataset verification (PDF resolvability) | `V2/tests/test_phase3_verification.py`, `data/processed/finqa_pdf_probe.json` | Confirm test PDFs in HF repo | 380/380 matched | M006 | No |
| 4 | Freeze 140 FinQA test questions | `V2/src/data/select_140.py` | Frozen TEST 140, seed 42 | `data/final/selected_140_questions.csv` | M007–M010 | No |
| 5 | Freeze FinQA DEV calibration set | `V2/src/data/select_calibration.py` | Frozen DEV 40, no TEST overlap | `data/calibration/calibration_questions.csv` | M011–M012 | No |
| 6 | Knowledge base (source PDFs) | `V2/src/retrieval/*`, `scripts/build_index.py` | PDFs → chunks → Chroma | 230 docs / 1239 chunks | M013–M021 | No |
| 7 | Qwen3-8B backend | `notebooks/colab_phase7_smoke.ipynb`, `src/models/llama_cpp_backend.py` | llama.cpp GGUF on Colab T4 | `phase7_smoke_test.json` | M022–M024 | No |
| 8 | Single-Agent RAG baseline | `colab_phase8_smoke.ipynb`, `src/rag/single_agent.py` | retrieve → generate | `phase8_smoke_test.json` | M025–M028 | No |
| 9 | Multi-Agent RAG | `colab_phase9_smoke.ipynb`, `src/rag/multi_agent.py` | retrieve → draft → verify | `phase9_smoke_test.json` | M029–M033 | No |
| 10 | Multi-Agent RAG + UQ / abstention | `colab_phase10_smoke.ipynb`, `src/rag/multi_agent_uq.py` | confidence gate ANSWER/ABSTAIN | `phase10_smoke_test.json` | M034–M036 | No |
| 11 | Streamlit live artefact | `colab_phase11_live.ipynb`, `app/streamlit_app.py` | live three-architecture UI | `app/streamlit_app.py` | M037–M039 | No |
| 12 | Pilot (18 cases) | `colab_phase12_pilot.ipynb`, `src/run/pilot.py` | 6×3=18, T smoke 0.55 NOT LOCKED | Phase 12 JSONL | M040–M041 | No |
| 13 | DEV calibration / threshold lock | `colab_phase13_calibration.ipynb`, `src/calibration/select.py` | lock T on DEV 40 | `threshold.lock.json` T=0.65 | M042–M045 | No |
| 14 | Benchmark runner / 9-case validation | `colab_phase14_benchmark_validation.ipynb`, `src/run/benchmark.py` | 3×3=9 engineering check | Phase 14 JSONL | M046–M048 | No |
| 15 | Final 420-case benchmark | `colab_phase15_full_benchmark.ipynb`, `scripts/run_full_benchmark.py` | official 140×3=420 | `phase15_.../cases.jsonl` | M049–M050 | No |
| 16 | Evaluation + LLM-as-judge | `colab_phase16_judge.ipynb`, `src/evaluation/` | CPU metrics + post-hoc judge | `phase16_cases.jsonl`, judge JSONL | M051–M054 | No |
| 17 | Statistics on frozen Phase 15/16 results | `scripts/run_statistics.py`, `src/statistics/` | McNemar / Wilcoxon / Spearman | `phase17_tests.csv` | M055–M056 | No |
| 18 | Qualitative error analysis | `src/error_analysis/taxonomy.py` | rule-based categories on 420 | `phase18_error_*.csv` | M057 | No |
| 19 | Reproducibility / research-integrity audit | `src/audit/checks.py` | read-only SHA/lock checks | `phase19_artefact_manifest.md` | M058 | No |
| 20 | Final live artefact | `app/streamlit_app.py`, `src/rag/live.py` | locked T=0.65 live pipelines | Streamlit pages | M059–M060 | No |
| 21 | Canonical final live-demo launcher | `notebooks/colab_phase21_final_live_demo.ipynb` | launch existing Streamlit | Colab proxy URL | M061 | Yes — launch only |


## PHASE 1 — Project foundation

No Colab notebook. Implementation: `V2/src/config/loader.py`, `V2/scripts/health_check.py`.

### MASTER CODE ID: M001

**Source file:** `V2/src/config/loader.py`
**Function/Class:** `load_experiment_config / ExperimentConfig / get_path`

### What this cell does

Loads `config/experiment.yaml` and resolves paths against the V2 root.

### Libraries used

PyYAML (`yaml.safe_load`), pathlib.

### Inputs

Optional override path; default `V2/config/experiment.yaml`.

### Outputs

`ExperimentConfig` with `.section()` / `.get()`; `get_path()` returns an absolute Path.

### Why this matters

Every later phase reads retrieval, model, dataset and path settings from this loader.

In [ ]:
# Copied from V2/src/config/loader.py
"""Central configuration loader for V2."""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import yaml

V2_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_EXPERIMENT_CONFIG = V2_ROOT / "config" / "experiment.yaml"
DEFAULT_PROMPTS_CONFIG = V2_ROOT / "config" / "prompts.yaml"
def project_root() -> Path:
    """Return the absolute path to the V2 project root."""
    return V2_ROOT
@dataclass(frozen=True)
class ExperimentConfig:
    """Immutable view of the loaded experiment configuration."""

    raw: dict[str, Any]
    source_path: Path

    def section(self, name: str) -> dict[str, Any]:
        value = self.raw.get(name, {})
        if value is None:
            return {}
        if not isinstance(value, dict):
            raise TypeError(f"Config section '{name}' must be a mapping, got {type(value)}")
        return value

    def get(self, *keys: str, default: Any = None) -> Any:
        node: Any = self.raw
        for key in keys:
            if not isinstance(node, dict) or key not in node:
                return default
            node = node[key]
        return node
def _load_yaml(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Configuration file not found: {path}")
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle)
    if data is None:
        return {}
    if not isinstance(data, dict):
        raise ValueError(f"Configuration root must be a mapping: {path}")
    return data
def load_experiment_config(path: Path | None = None) -> ExperimentConfig:
    """Load experiment.yaml (or an override path)."""
    config_path = Path(path) if path is not None else DEFAULT_EXPERIMENT_CONFIG
    return ExperimentConfig(raw=_load_yaml(config_path), source_path=config_path.resolve())
def load_prompts_config(path: Path | None = None) -> dict[str, Any]:
    """Load prompts.yaml (placeholders in Phase 1)."""
    config_path = Path(path) if path is not None else DEFAULT_PROMPTS_CONFIG
    return _load_yaml(config_path)
def get_path(config: ExperimentConfig, key: str) -> Path:
    """Resolve a configured relative path against the V2 root."""
    paths = config.section("paths")
    if key not in paths:
        raise KeyError(f"Unknown path key '{key}'. Available: {sorted(paths)}")
    relative = Path(str(paths[key]))
    if relative.is_absolute():
        return relative
    return (V2_ROOT / relative).resolve()


### Viva explanation

This is the central config loader. It does not invent unverified scientific settings; those stay in YAML, including null threshold until the lock file exists.

### Likely viva question

Where does the project load experiment.yaml?

### Answer

`V2/src/config/loader.py`, `load_experiment_config()` (this cell).

### MASTER CODE ID: M002

**Source file:** `V2/src/utils/run_id.py`
**Function/Class:** `create_run_id`

### What this cell does

Builds `{prefix}_{UTC}_{short-uuid}` run IDs used in raw result folders.

### Libraries used

datetime, uuid, re.

### Inputs

`prefix` string, e.g. `phase15`.

### Outputs

A unique run_id string.

### Why this matters

Raw JSONL lives under `results/raw/.../{run_id}/` so runs are not overwritten.

In [ ]:
# Copied from V2/src/utils/run_id.py
"""Run identification helpers for reproducible experiment tracking."""
from __future__ import annotations
from datetime import datetime, timezone
import re
import uuid

def create_run_id(prefix: str = "run") -> str:
    """Create a unique run ID: ``{prefix}_{UTC-timestamp}_{short-uuid}``."""
    safe_prefix = re.sub(r"[^a-zA-Z0-9_-]+", "-", prefix).strip("-") or "run"
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    short = uuid.uuid4().hex[:8]
    return f"{safe_prefix}_{stamp}_{short}"


### Viva explanation

Every official job gets a run_id. Phase 15’s official folder is `phase15_20260826T203744Z_dae9c3a4`.

### Likely viva question

How are experiment runs identified?

### Answer

`create_run_id()` in `V2/src/utils/run_id.py`.

### MASTER CODE ID: M003

**Source file:** `V2/scripts/health_check.py`
**Function/Class:** `main`

### What this cell does

Phase 1 CLI: load config, create a phase1 run_id, print OK. Does not touch V1.

### Libraries used

src.config, src.utils logging.

### Inputs

None (uses V2 root).

### Outputs

Prints `OK | V2_ROOT=... | run_id=...`; return code 0.

### Why this matters

Confirms the V2 package imports before any RAG code existed.

In [ ]:
# Copied from V2/scripts/health_check.py
"""Phase 1 health check — verify V2 config loads without touching V1."""
from __future__ import annotations
import sys
from pathlib import Path
from src.config import load_experiment_config, project_root
from src.utils import create_run_id, get_logger, setup_logging

def main() -> int:
    root = project_root()
    config = load_experiment_config()
    run_id = create_run_id("phase1")
    setup_logging(level="INFO", console=True, file=False)
    log = get_logger(run_id=run_id, phase="phase1")
    log.info(
        "health_check ok root=%s project=%s dataset=%s",
        root,
        config.get("project", "name"),
        config.get("dataset", "subset"),
    )
    print(f"OK | V2_ROOT={root} | run_id={run_id}")
    return 0


### Viva explanation

Phase 1 only proved the skeleton: config loads, V1 is untouched, no dataset freeze yet.

### Likely viva question

What did Phase 1 actually implement?

### Answer

V2 tree, YAML config loader, run IDs, health check — not RAG.

## PHASE 2 — V1 audit + FinQA live profile

### MASTER CODE ID: M004

**Source file:** `V2/src/data/profile_finqa.py`
**Function/Class:** `HF_DATASET_ID / load_finqa`

### What this cell does

Live-loads T²-RAGBench FinQA via Hugging Face `datasets`. Does not select the 140.

### Libraries used

`datasets.load_dataset`.

### Inputs

Hugging Face id `G4KMU/t2-ragbench`, subset `FinQA`.

### Outputs

A DatasetDict with train/dev/test splits.

### Why this matters

This is the actual dataset family: FinQA only, from T²-RAGBench.

In [ ]:
# Copied from V2/src/data/profile_finqa.py
"""Profile T²-RAGBench FinQA without selecting the frozen 140 set."""
from __future__ import annotations
from collections import Counter
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any

HF_DATASET_ID = "G4KMU/t2-ragbench"
HF_SUBSET = "FinQA"
EXPECTED_COLUMNS = [
    "id",
    "context_id",
    "split",
    "question",
    "program_answer",
    "original_answer",
    "context",
    "table",
    "pre_text",
    "post_text",
    "file_name",
    "company_name",
    "company_symbol",
    "report_year",
    "page_number",
    "company_sector",
    "company_industry",
    "company_headquarters",
    "company_date_added",
    "company_cik",
    "company_founded",
]
def load_finqa():
    from datasets import load_dataset

    return load_dataset(HF_DATASET_ID, HF_SUBSET)


### Viva explanation

The evaluation dataset is only the FinQA subset of T²-RAGBench. Phase 2 recorded splits 6251/883/1147.

### Likely viva question

Which dataset does V2 use?

### Answer

`G4KMU/t2-ragbench`, subset FinQA, loaded in `load_finqa()`.

### MASTER CODE ID: M005

**Source file:** `V2/src/data/profile_finqa.py`
**Function/Class:** `check_pdf_availability`

### What this cell does

Phase 2 note: `load_dataset()` returns tabular fields; PDFs sit in the HF repo tree, not as row blobs.

### Libraries used

Optional `huggingface_hub`.

### Inputs

Dataset info object and observed `file_name` values.

### Outputs

A probe dict (no full PDF download).

### Why this matters

Stops V1’s gold-context-as-document mistake: KB must be source PDFs.

In [ ]:
# Copied from V2/src/data/profile_finqa.py
"""Profile T²-RAGBench FinQA without selecting the frozen 140 set."""
from __future__ import annotations
from collections import Counter
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any

def check_pdf_availability(dataset_info: Any, file_names: set[str]) -> dict[str, Any]:
    """Inspect whether PDF paths are advertised; do not download the full PDF tree here."""
    result: dict[str, Any] = {
        "dataset_card_claims_pdfs": True,
        "pdfs_bundled_in_arrow_rows": False,
        "unique_file_names_observed": len(file_names),
        "example_file_names": sorted(file_names)[:10],
        "local_pdf_probe": {},
        "notes": [],
    }

    # Hugging Face datasets load for FinQA returns tabular fields only; PDFs are in the
    # repository data tree (clone) according to the dataset card — not as row blobs.
    result["notes"].append(
        "Official card: clone G4KMU/t2-ragbench to obtain PDFs under data/ organised by "
        "dataset/split. load_dataset() returns text/metadata columns, not PDF bytes."
    )

    # Optional local cache probe (HF datasets cache) — informational only.
    try:
        from huggingface_hub import hf_hub_url  # type: ignore

        _ = hf_hub_url
        result["notes"].append("huggingface_hub is available for later PDF fetch/clone.")
    except Exception:
        result["notes"].append("huggingface_hub not imported; PDF clone still required later.")

    return result


### Viva explanation

Gold `context` is evaluation-only. Retrieval documents are annual-report page PDFs.

### Likely viva question

Are gold contexts stored in the vector index?

### Answer

No. Phase 2/6 treat gold context as oracle material, not KB text.

## PHASE 3 — Dataset verification (PDF resolvability)

### MASTER CODE ID: M006

**Source file:** `V2/tests/test_phase3_verification.py`
**Function/Class:** `test_phase3_pdf_probe_test_fully_resolved`

### What this cell does

Asserts the saved PDF probe: 380/380 test `file_name` values resolve; 140 not frozen yet.

### Libraries used

json, pathlib.

### Inputs

`V2/data/processed/finqa_pdf_probe.json`.

### Outputs

Pass/fail assertions; mapping rule `data/FinQA/{split}/{file_name}`.

### Why this matters

Phase 3 closed PDF resolvability before any download in Phase 6.

In [ ]:
# Copied from V2/tests/test_phase3_verification.py
"""Phase 3 tests: verification artefacts exist; 140 not frozen."""
from __future__ import annotations
import json
from pathlib import Path
from src.config import project_root

def test_phase3_pdf_probe_test_fully_resolved() -> None:
    probe = json.loads(
        (project_root() / "data" / "processed" / "finqa_pdf_probe.json").read_text(encoding="utf-8")
    )
    assert probe["phase"] == 3
    assert probe["phase3_selected_140"] is False
    assert probe["test_pdf_resolution"]["matched_in_repo"] == 380
    assert probe["test_pdf_resolution"]["missing"] == 0
    assert probe["path_mapping_rule"] == "repo_path = data/FinQA/{split}/{file_name}"


### Viva explanation

Test split has 380 unique PDFs and all 380 were present in the HF repo. The 140 freeze is Phase 4.

### Likely viva question

Did you verify source PDFs exist before building the KB?

### Answer

Yes — Phase 3 probe `finqa_pdf_probe.json`, asserted here.

## PHASE 4 — Freeze 140 FinQA test questions

### MASTER CODE ID: M007

**Source file:** `V2/src/data/select_140.py`
**Function/Class:** `is_essential_eligible / filter_and_dedupe`

### What this cell does

Drops rows missing id/question/program_answer/context_id/file_name/context; dedupes normalised questions.

### Libraries used

stdlib only.

### Inputs

FinQA test rows from `load_dataset`.

### Outputs

Eligible unique rows + filter stats (1147 → 1144 eligible).

### Why this matters

The frozen 140 is a subset of essential, de-duplicated test questions.

In [ ]:
# Copied from V2/src/data/select_140.py
"""Phase 4: filter FinQA test split and freeze a reproducible 140-question set."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any

def normalize_question(text: str) -> str:
    return " ".join(str(text).strip().lower().split())
def is_essential_eligible(row: dict[str, Any]) -> bool:
    if _is_empty(row.get("id")):
        return False
    if _is_empty(row.get("question")):
        return False
    if _is_empty(row.get("program_answer")):
        # Primary evaluation field must exist for the frozen test set.
        return False
    if _is_empty(row.get("context_id")):
        return False
    if _is_empty(row.get("file_name")):
        return False
    if _is_empty(row.get("context")):
        return False
    return True
def filter_and_dedupe(rows: list[dict[str, Any]], split: str = "test") -> dict[str, Any]:
    """Filter malformed rows and dedupe by normalized question (keep lowest id)."""
    stats = {
        "input_rows": len(rows),
        "dropped_not_essential": 0,
        "dropped_duplicate_question": 0,
    }
    eligible: list[dict[str, Any]] = []
    for row in rows:
        if not is_essential_eligible(row):
            stats["dropped_not_essential"] += 1
            continue
        eligible.append(row_to_record(dict(row), split))

    # Deterministic dedupe: sort by id, keep first occurrence of normalized question.
    eligible.sort(key=lambda r: str(r["id"]))
    seen_questions: set[str] = set()
    unique: list[dict[str, Any]] = []
    for row in eligible:
        key = normalize_question(str(row["question"]))
        if key in seen_questions:
            stats["dropped_duplicate_question"] += 1
            continue
        seen_questions.add(key)
        unique.append(row)

    stats["eligible_unique_questions"] = len(unique)
    return {"rows": unique, "stats": stats}


### Viva explanation

Eligibility requires `program_answer` because numeric match is the primary correctness metric.

### Likely viva question

What makes a FinQA row eligible for the 140?

### Answer

Non-empty id, question, program_answer, context_id, file_name, context; then question dedupe.

### MASTER CODE ID: M008

**Source file:** `V2/src/data/select_140.py`
**Function/Class:** `stratified_sample`

### What this cell does

Seeded shuffle (seed 42), greedy caps max 3/company and 1/file, then freeze by id.

### Libraries used

random, collections.Counter.

### Inputs

Eligible test rows, n=140, seed=42.

### Outputs

140 rows, 77 companies, 140 files (from the freeze manifest).

### Why this matters

Stops one company dominating the 420-case comparison.

In [ ]:
# Copied from V2/src/data/select_140.py
"""Phase 4: filter FinQA test split and freeze a reproducible 140-question set."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any

def stratified_sample(
    rows: list[dict[str, Any]],
    *,
    n: int = 140,
    seed: int = 42,
    max_per_company: int = 3,
    max_per_file: int = 1,
) -> dict[str, Any]:
    """Greedy stratified sample with company and file caps.

    Rows are shuffled with a seeded RNG, then accepted if they do not exceed
    ``max_per_company`` (by company_symbol, fallback company_name) or
    ``max_per_file`` (by file_name). Caps relax once if n cannot be reached.
    """
    if n > len(rows):
        raise ValueError(f"Requested n={n} exceeds eligible pool size {len(rows)}")

    rng = random.Random(seed)
    order = list(rows)
    rng.shuffle(order)

    def _company_key(row: dict[str, Any]) -> str:
        symbol = str(row.get("company_symbol") or "").strip()
        if symbol:
            return f"sym:{symbol}"
        name = str(row.get("company_name") or "").strip()
        if name:
            return f"name:{name}"
        return "unknown"

    def _select(cap_company: int, cap_file: int) -> list[dict[str, Any]]:
        selected: list[dict[str, Any]] = []
        company_counts: Counter[str] = Counter()
        file_counts: Counter[str] = Counter()
        for row in order:
            if len(selected) >= n:
                break
            company = _company_key(row)
            file_name = str(row.get("file_name") or "")
            if company_counts[company] >= cap_company:
                continue
            if file_counts[file_name] >= cap_file:
                continue
            selected.append(row)
            company_counts[company] += 1
            file_counts[file_name] += 1
        return selected

    caps_used = {"max_per_company": max_per_company, "max_per_file": max_per_file}
    selected = _select(max_per_company, max_per_file)
    if len(selected) < n:
        caps_used = {"max_per_company": max_per_company + 1, "max_per_file": max_per_file + 1}
        selected = _select(max_per_company + 1, max_per_file + 1)
    if len(selected) < n:
        caps_used = {"max_per_company": max_per_company + 2, "max_per_file": max_per_file + 2}
        selected = _select(max_per_company + 2, max_per_file + 2)
    if len(selected) < n:
        raise RuntimeError(
            f"Could only select {len(selected)}/{n} rows under diversity caps {caps_used}"
        )

    # Stable output order by id for readable diffs; selection set is what matters.
    selected_sorted = sorted(selected, key=lambda r: str(r["id"]))
    company_dist = Counter(_company_key(r) for r in selected_sorted)
    file_dist = Counter(str(r.get("file_name") or "") for r in selected_sorted)
    year_dist = Counter(str(r.get("report_year") or "") for r in selected_sorted)

    return {
        "rows": selected_sorted,
        "seed": seed,
        "n": len(selected_sorted),
        "caps_used": caps_used,
        "unique_companies": len(company_dist),
        "unique_files": len(file_dist),
        "max_questions_per_company": max(company_dist.values()) if company_dist else 0,
        "max_questions_per_file": max(file_dist.values()) if file_dist else 0,
        "company_distribution": dict(sorted(company_dist.items())),
        "report_year_distribution": dict(sorted(year_dist.items())),
    }


### Viva explanation

Sampling is frozen. I do not resample after seeing results. Seed 42 is in the manifest SHA.

### Likely viva question

How was the 140 sampled?

### Answer

`stratified_sample()` in `select_140.py`: seed 42, company/file caps.

### MASTER CODE ID: M009

**Source file:** `V2/src/data/select_140.py`
**Function/Class:** `freeze_test_140`

### What this cell does

Writes the frozen CSV and sampling manifest; tells later phases not to alter the freeze.

### Libraries used

json, datetime.

### Inputs

Filtered test rows; output CSV/manifest paths.

### Outputs

`data/final/selected_140_questions.csv` and `sampling_manifest.json`.

### Why this matters

Phase 15 IDs must match this manifest SHA.

In [ ]:
# Copied from V2/src/data/select_140.py
"""Phase 4: filter FinQA test split and freeze a reproducible 140-question set."""
from __future__ import annotations
from collections import Counter, defaultdict
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any

def freeze_test_140(
    test_rows: list[dict[str, Any]],
    *,
    output_csv: Path,
    output_manifest: Path,
    n: int = 140,
    seed: int = 42,
    max_per_company: int = 3,
    max_per_file: int = 1,
    dataset_id: str = "G4KMU/t2-ragbench",
    subset: str = "FinQA",
) -> dict[str, Any]:
    filtered = filter_and_dedupe(test_rows, split="test")
    sampled = stratified_sample(
        filtered["rows"],
        n=n,
        seed=seed,
        max_per_company=max_per_company,
        max_per_file=max_per_file,
    )
    write_csv(output_csv, sampled["rows"])
    manifest = {
        "phase": 4,
        "frozen": True,
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_id": dataset_id,
        "subset": subset,
        "source_split": "test",
        "n": sampled["n"],
        "sampling_seed": seed,
        "filter_rules": {
            "require_non_empty": [
                "id",
                "question",
                "program_answer",
                "context_id",
                "file_name",
                "context",
            ],
            "dedupe": "normalize_question keep_lowest_id",
        },
        "diversity_caps_requested": {
            "max_per_company": max_per_company,
            "max_per_file": max_per_file,
        },
        "diversity_caps_used": sampled["caps_used"],
        "filter_stats": filtered["stats"],
        "unique_companies": sampled["unique_companies"],
        "unique_files": sampled["unique_files"],
        "max_questions_per_company": sampled["max_questions_per_company"],
        "max_questions_per_file": sampled["max_questions_per_file"],
        "report_year_distribution": sampled["report_year_distribution"],
        "selected_ids": [r["id"] for r in sampled["rows"]],
        "selected_ids_sha256": rows_fingerprint(sampled["rows"]),
        "output_csv": str(output_csv),
        "note": (
            "Do not alter this freeze because of experimental results. "
            "Calibration data is selected separately from FinQA dev (Phase 5)."
        ),
    }
    write_manifest(output_manifest, manifest)
    return {"rows": sampled["rows"], "manifest": manifest}


### Viva explanation

The freeze note is explicit: do not change this set because of experimental results.

### Likely viva question

Where is the frozen TEST set written?

### Answer

`freeze_test_140()` → `V2/data/final/selected_140_questions.csv`.

### MASTER CODE ID: M010

**Source file:** `V2/src/run/subset.py`
**Function/Class:** `load_frozen_question_rows`

### What this cell does

Read-only loader used by pilot/benchmark/live catalogue. Does not rewrite the CSV.

### Libraries used

csv.

### Inputs

Frozen TEST CSV path.

### Outputs

List of dicts with id, question, program_answer, file_name, company_symbol.

### Why this matters

Phases 12–15 and Streamlit all read the same freeze.

In [ ]:
# Copied from V2/src/run/subset.py
"""Reproducible Phase 12 pilot subset from the frozen 140-question test set."""
from __future__ import annotations
import csv
import hashlib
import json
from pathlib import Path
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root

PILOT_N_QUESTIONS = 6
def frozen_test_csv(config: ExperimentConfig | None = None) -> Path:
    cfg = config or load_experiment_config()
    dataset = cfg.section("dataset")
    rel = str(dataset.get("frozen_test_set") or "data/final/selected_140_questions.csv")
    path = (project_root() / rel).resolve()
    if not path.is_file():
        path = get_path(cfg, "data_final") / "selected_140_questions.csv"
    return path
def load_frozen_question_rows(csv_path: Path | None = None) -> list[dict[str, str]]:
    path = csv_path or frozen_test_csv()
    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            qid = str(row.get("id") or "").strip()
            question = str(row.get("question") or "").strip()
            if not qid or not question:
                continue
            rows.append(
                {
                    "id": qid,
                    "question": question,
                    "program_answer": str(row.get("program_answer") or ""),
                    "original_answer": str(row.get("original_answer") or ""),
                    "file_name": str(row.get("file_name") or ""),
                    "company_symbol": str(row.get("company_symbol") or ""),
                }
            )
    return rows


### Viva explanation

Later jobs never resample. They load this CSV.

### Likely viva question

How do later phases get the 140 questions?

### Answer

`load_frozen_question_rows()` in `V2/src/run/subset.py`.

## PHASE 5 — Freeze FinQA DEV calibration set

### MASTER CODE ID: M011

**Source file:** `V2/src/data/select_calibration.py`
**Function/Class:** `exclude_test_overlap`

### What this cell does

Drops DEV rows whose id or normalised question overlaps the frozen TEST 140.

### Libraries used

csv via `load_frozen_test_ids_and_questions`.

### Inputs

DEV rows + forbidden TEST ids/questions.

### Outputs

Kept DEV rows and overlap counts.

### Why this matters

RQ3 threshold must not be tuned on TEST.

In [ ]:
# Copied from V2/src/data/select_calibration.py
"""Phase 5: freeze FinQA DEV calibration questions (separate from the frozen test 140)."""
from __future__ import annotations
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from typing import Any
from src.data.select_140 import (
    EXPORT_COLUMNS,
    filter_and_dedupe,
    normalize_question,
    stratified_sample,
    write_csv,
)

def load_frozen_test_ids_and_questions(
    test_csv: Path,
) -> tuple[set[str], set[str]]:
    if not test_csv.exists():
        raise FileNotFoundError(
            f"Frozen test set not found: {test_csv}. Run Phase 4 first."
        )
    ids: set[str] = set()
    questions: set[str] = set()
    with test_csv.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            ids.add(str(row["id"]))
            questions.add(normalize_question(str(row["question"])))
    return ids, questions
def exclude_test_overlap(
    rows: list[dict[str, Any]],
    forbidden_ids: set[str],
    forbidden_questions: set[str],
) -> dict[str, Any]:
    kept: list[dict[str, Any]] = []
    dropped_id = 0
    dropped_question = 0
    for row in rows:
        if str(row["id"]) in forbidden_ids:
            dropped_id += 1
            continue
        if normalize_question(str(row["question"])) in forbidden_questions:
            dropped_question += 1
            continue
        kept.append(row)
    return {
        "rows": kept,
        "stats": {
            "input_rows": len(rows),
            "dropped_id_overlap": dropped_id,
            "dropped_question_overlap": dropped_question,
            "remaining": len(kept),
        },
    }


### Viva explanation

Calibration is FinQA **dev** 40. The lock file records `used_frozen_test_140: false`.

### Likely viva question

Could the threshold have been tuned on the 140?

### Answer

No. Phase 5 excludes TEST overlap; Phase 13 lock requires `source_split=dev`.

### MASTER CODE ID: M012

**Source file:** `V2/src/data/select_calibration.py`
**Function/Class:** `freeze_calibration`

### What this cell does

Samples 40 DEV questions (seed 42, max 2/company, max 1/file) and writes a manifest with `threshold_locked: False`.

### Libraries used

same sampling helpers as Phase 4.

### Inputs

DEV rows + frozen TEST CSV.

### Outputs

`data/calibration/calibration_questions.csv`. Threshold is NOT locked here.

### Why this matters

Phase 5 only freezes the calibration sample. T is locked in Phase 13.

In [ ]:
# Copied from V2/src/data/select_calibration.py
"""Phase 5: freeze FinQA DEV calibration questions (separate from the frozen test 140)."""
from __future__ import annotations
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
from typing import Any
from src.data.select_140 import (
    EXPORT_COLUMNS,
    filter_and_dedupe,
    normalize_question,
    stratified_sample,
    write_csv,
)

def freeze_calibration(
    dev_rows: list[dict[str, Any]],
    *,
    frozen_test_csv: Path,
    output_csv: Path,
    output_manifest: Path,
    n: int = 40,
    seed: int = 42,
    max_per_company: int = 2,
    max_per_file: int = 1,
    dataset_id: str = "G4KMU/t2-ragbench",
    subset: str = "FinQA",
) -> dict[str, Any]:
    forbidden_ids, forbidden_questions = load_frozen_test_ids_and_questions(frozen_test_csv)
    filtered = filter_and_dedupe(dev_rows, split="dev")
    cleaned = exclude_test_overlap(filtered["rows"], forbidden_ids, forbidden_questions)
    sampled = stratified_sample(
        cleaned["rows"],
        n=n,
        seed=seed,
        max_per_company=max_per_company,
        max_per_file=max_per_file,
    )
    write_csv(output_csv, sampled["rows"])

    # Hard safety checks before writing the manifest.
    selected_ids = [r["id"] for r in sampled["rows"]]
    selected_questions = {normalize_question(str(r["question"])) for r in sampled["rows"]}
    if set(selected_ids) & forbidden_ids:
        raise RuntimeError("Calibration set overlaps frozen test ids")
    if selected_questions & forbidden_questions:
        raise RuntimeError("Calibration set overlaps frozen test questions")

    manifest = {
        "phase": 5,
        "frozen": True,
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": "confidence_threshold_calibration_only",
        "threshold_locked": False,
        "dataset_id": dataset_id,
        "subset": subset,
        "source_split": "dev",
        "n": sampled["n"],
        "sampling_seed": seed,
        "filter_rules": {
            "require_non_empty": [
                "id",
                "question",
                "program_answer",
                "context_id",
                "file_name",
                "context",
            ],
            "dedupe": "normalize_question keep_lowest_id",
            "exclude_overlap_with": str(frozen_test_csv),
        },
        "diversity_caps_requested": {
            "max_per_company": max_per_company,
            "max_per_file": max_per_file,
        },
        "diversity_caps_used": sampled["caps_used"],
        "filter_stats": filtered["stats"],
        "overlap_exclusion_stats": cleaned["stats"],
        "unique_companies": sampled["unique_companies"],
        "unique_files": sampled["unique_files"],
        "max_questions_per_company": sampled["max_questions_per_company"],
        "max_questions_per_file": sampled["max_questions_per_file"],
        "report_year_distribution": sampled["report_year_distribution"],
        "selected_ids": selected_ids,
        "selected_ids_sha256": rows_fingerprint(sampled["rows"]),
        "frozen_test_csv": str(frozen_test_csv),
        "output_csv": str(output_csv),
        "export_columns": EXPORT_COLUMNS,
        "note": (
            "Use this set only to choose confidence method/threshold. "
            "Lock threshold before evaluating the frozen test 140. "
            "Do not tune the threshold on the test set."
        ),
    }
    output_manifest.parent.mkdir(parents=True, exist_ok=True)
    output_manifest.write_text(json.dumps(manifest, indent=2, default=str) + "\n", encoding="utf-8")
    return {"rows": sampled["rows"], "manifest": manifest}


### Viva explanation

If asked ‘when was 0.65 chosen?’ — not Phase 5. Phase 5 only froze the 40 DEV questions.

### Likely viva question

What did Phase 5 lock?

### Answer

The 40 DEV questions, not T. `threshold_locked` is False in that manifest.

## PHASE 6 — Knowledge base (source PDFs)

Retrieval walkthrough: obtain PDFs → extract → chunk → embed → Chroma → query.

### MASTER CODE ID: M013

**Source file:** `V2/src/retrieval/pdf_fetch.py`
**Function/Class:** `CorpusDoc / collect_corpus_targets`

### What this cell does

Builds the PDF list: frozen TEST files + DEV calibration files + 50 train distractors (seed 42).

### Libraries used

huggingface `datasets` for distractors; csv for freezes.

### Inputs

TEST CSV, calibration CSV, distractor_count=50.

### Outputs

List of `CorpusDoc` with `repo_path = data/FinQA/{split}/{file_name}`.

### Why this matters

This is where documents are obtained. Gold context is not a document.

In [ ]:
# Copied from V2/src/retrieval/pdf_fetch.py
"""Download FinQA page PDFs from the Hugging Face dataset repo."""
from __future__ import annotations
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from huggingface_hub import hf_hub_download

@dataclass(frozen=True)
class CorpusDoc:
    split: str
    file_name: str
    role: str  # test | calibration | distractor
    company_symbol: str = ""
    company_name: str = ""
    report_year: str = ""
    context_id: str = ""
    question_id: str = ""

    @property
    def repo_path(self) -> str:
        return f"data/FinQA/{self.split}/{self.file_name}"

    @property
    def doc_key(self) -> str:
        return f"{self.split}::{self.file_name}"
def collect_corpus_targets(
    *,
    test_csv: Path,
    calibration_csv: Path,
    distractor_count: int = 50,
    distractor_seed: int = 42,
    dataset_id: str = "G4KMU/t2-ragbench",
    subset: str = "FinQA",
) -> list[CorpusDoc]:
    """Collect unique PDFs for test + calibration, plus optional train distractors."""
    core = _dedupe_by_doc_key(
        [
            *_read_csv_docs(test_csv, split="test", role="test"),
            *_read_csv_docs(calibration_csv, split="dev", role="calibration"),
        ]
    )
    core_files = {d.file_name for d in core}
    if distractor_count <= 0:
        return core

    from datasets import load_dataset
    import random

    ds = load_dataset(dataset_id, subset, split="train")
    candidates: list[CorpusDoc] = []
    seen_files: set[str] = set(core_files)
    for row in ds:
        file_name = str(row.get("file_name") or "").strip()
        if not file_name or file_name in seen_files:
            continue
        seen_files.add(file_name)
        candidates.append(
            CorpusDoc(
                split="train",
                file_name=file_name,
                role="distractor",
                company_symbol=str(row.get("company_symbol") or ""),
                company_name=str(row.get("company_name") or ""),
                report_year=str(row.get("report_year") or ""),
                context_id=str(row.get("context_id") or ""),
                question_id=str(row.get("id") or ""),
            )
        )
    rng = random.Random(distractor_seed)
    rng.shuffle(candidates)
    return core + candidates[:distractor_count]


### Viva explanation

The corpus is 140 test PDFs + 40 calibration PDFs + 50 train distractors, then de-duplicated by file. Config records 230 indexed docs.

### Likely viva question

Where are documents collected?

### Answer

`collect_corpus_targets()` in `V2/src/retrieval/pdf_fetch.py`.

### MASTER CODE ID: M014

**Source file:** `V2/src/retrieval/pdf_fetch.py`
**Function/Class:** `download_pdfs`

### What this cell does

Downloads each page PDF from the HF dataset repo into `knowledge_base/documents/{split}/`.

### Libraries used

`huggingface_hub.hf_hub_download`.

### Inputs

`CorpusDoc` list, documents_dir, repo_id `G4KMU/t2-ragbench`.

### Outputs

local_paths map + download/skip/fail counts.

### Why this matters

Phase 6 actually fetches bytes. This notebook will not call it (`REBUILD_KB=False`).

In [ ]:
# Copied from V2/src/retrieval/pdf_fetch.py
"""Download FinQA page PDFs from the Hugging Face dataset repo."""
from __future__ import annotations
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from huggingface_hub import hf_hub_download

def download_pdfs(
    docs: list[CorpusDoc],
    *,
    documents_dir: Path,
    repo_id: str = "G4KMU/t2-ragbench",
) -> dict:
    """Download page PDFs into ``documents_dir/{split}/{file_name}``."""
    documents_dir.mkdir(parents=True, exist_ok=True)
    downloaded = 0
    skipped = 0
    failed: list[dict] = []
    local_paths: dict[str, str] = {}

    for doc in docs:
        dest = documents_dir / doc.split / doc.file_name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.stat().st_size > 0:
            skipped += 1
            local_paths[doc.doc_key] = str(dest)
            continue
        try:
            cached = hf_hub_download(
                repo_id=repo_id,
                repo_type="dataset",
                filename=doc.repo_path,
            )
            dest.write_bytes(Path(cached).read_bytes())
            downloaded += 1
            local_paths[doc.doc_key] = str(dest)
        except Exception as exc:  # noqa: BLE001 — record and continue
            failed.append({"doc_key": doc.doc_key, "repo_path": doc.repo_path, "error": str(exc)})

    return {
        "requested": len(docs),
        "downloaded": downloaded,
        "skipped_existing": skipped,
        "failed": failed,
        "local_paths": local_paths,
    }


### Viva explanation

PDFs are Hugging Face dataset files, not scraped websites.

### Likely viva question

How are PDFs downloaded?

### Answer

`download_pdfs()` using `hf_hub_download` on `doc.repo_path`.

### MASTER CODE ID: M015

**Source file:** `V2/src/retrieval/extract.py`
**Function/Class:** `extract_pdf_pages / clean_text`

### What this cell does

PyMuPDF text extraction per page; null bytes stripped; empty pages dropped.

### Libraries used

PyMuPDF (`fitz`).

### Inputs

A local PDF path.

### Outputs

List of `{text, page, local_path}`.

### Why this matters

This is the PDF extraction implementation.

In [ ]:
# Copied from V2/src/retrieval/extract.py
"""Extract text from FinQA page PDFs."""
from __future__ import annotations
from pathlib import Path
from typing import Any

def clean_text(text: str) -> str:
    return " ".join(str(text).replace("\x00", " ").split())
def extract_pdf_pages(pdf_path: Path) -> list[dict[str, Any]]:
    """Return one page dict per PDF page with text + base metadata."""
    try:
        import fitz  # PyMuPDF
    except ImportError as exc:
        raise RuntimeError("Install PyMuPDF: pip install pymupdf") from exc

    pages: list[dict[str, Any]] = []
    document = fitz.open(pdf_path)
    try:
        for page_index, page in enumerate(document, start=1):
            text = clean_text(page.get_text("text"))
            if not text:
                continue
            pages.append(
                {
                    "text": text,
                    "page": page_index,
                    "local_path": str(pdf_path),
                }
            )
    finally:
        document.close()
    return pages


### Viva explanation

I used PyMuPDF, not OCR. These are already born-digital annual-report pages.

### Likely viva question

Where is PDF extraction?

### Answer

`extract_pdf_pages()` in `V2/src/retrieval/extract.py`.

### MASTER CODE ID: M016

**Source file:** `V2/src/retrieval/chunking.py`
**Function/Class:** `split_text / chunk_pages`

### What this cell does

Character windows of size 900 with overlap 150; each chunk keeps PDF provenance metadata.

### Libraries used

stdlib.

### Inputs

Page dicts; chunk_size=900, chunk_overlap=150 from `experiment.yaml`.

### Outputs

`{text, metadata}` chunks with `source_type='pdf'`.

### Why this matters

This is the chunking implementation. Config records 1239 searchable chunks after indexing.

In [ ]:
# Copied from V2/src/retrieval/chunking.py
"""Simple character chunking for KB indexing."""
from __future__ import annotations
from typing import Any

def split_text(text: str, chunk_size: int = 900, chunk_overlap: int = 150) -> list[str]:
    """Sliding-window character chunks with overlap."""
    text = " ".join(str(text).split())
    if not text:
        return []
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be >= 0 and < chunk_size")
    if len(text) <= chunk_size:
        return [text]

    chunks: list[str] = []
    start = 0
    step = chunk_size - chunk_overlap
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks
def chunk_pages(
    pages: list[dict[str, Any]],
    *,
    chunk_size: int = 900,
    chunk_overlap: int = 150,
    base_metadata: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    """Chunk page texts; each chunk carries provenance metadata."""
    base_metadata = base_metadata or {}
    chunks: list[dict[str, Any]] = []
    for page in pages:
        pieces = split_text(page["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        for idx, piece in enumerate(pieces):
            meta = {
                **base_metadata,
                "page": int(page.get("page") or 1),
                "local_path": str(page.get("local_path") or ""),
                "chunk_index": idx,
                "source_type": "pdf",
            }
            chunks.append({"text": piece, "metadata": meta})
    return chunks


### Viva explanation

Chunking is character-based, not tokens. Overlap 150 keeps numbers from splitting at window edges.

### Likely viva question

Where is chunk size set?

### Answer

Defaults in `split_text`/`build_knowledge_base` and `retrieval.chunk_size: 900` in `experiment.yaml`.

### MASTER CODE ID: M017

**Source file:** `V2/src/retrieval/embeddings.py`
**Function/Class:** `embed_texts / get_embedding_model`

### What this cell does

Encodes chunk text with `BAAI/bge-small-en-v1.5` via sentence-transformers, normalised vectors.

### Libraries used

sentence-transformers.

### Inputs

List of chunk strings; model name from config.

### Outputs

List of embedding vectors.

### Why this matters

Same embedder is used at index time and query time.

In [ ]:
# Copied from V2/src/retrieval/embeddings.py
"""Embedding model loader for the knowledge base."""
from __future__ import annotations
from functools import lru_cache
from typing import Any

@lru_cache(maxsize=2)
def get_embedding_model(model_name: str = "BAAI/bge-small-en-v1.5") -> Any:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise RuntimeError(
            "Install sentence-transformers: pip install sentence-transformers"
        ) from exc
    return SentenceTransformer(model_name)
def embed_texts(
    texts: list[str],
    *,
    model_name: str = "BAAI/bge-small-en-v1.5",
    batch_size: int = 32,
) -> list[list[float]]:
    model = get_embedding_model(model_name)
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=len(texts) > 64,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return [vector.tolist() for vector in vectors]


### Viva explanation

The embedding model is BGE-small, not the LLM. Qwen never embeds the KB.

### Likely viva question

Where are embeddings created?

### Answer

`embed_texts()` in `V2/src/retrieval/embeddings.py`.

### MASTER CODE ID: M018

**Source file:** `V2/src/retrieval/index.py`
**Function/Class:** `_chroma_client / load_collection`

### What this cell does

Creates a persistent Chroma client and collection `finqa_source_pdfs` with cosine HNSW space.

### Libraries used

chromadb.

### Inputs

`persist_dir` = `knowledge_base/index`.

### Outputs

Chroma client / collection.

### Why this matters

This is where ChromaDB is initialised.

In [ ]:
# Copied from V2/src/retrieval/index.py
"""Build and load the persistent Chroma knowledge-base index."""
from __future__ import annotations
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any
from src.retrieval.chunking import chunk_pages
from src.retrieval.embeddings import embed_texts
from src.retrieval.extract import extract_pdf_pages
from src.retrieval.pdf_fetch import CorpusDoc

COLLECTION_NAME = "finqa_source_pdfs"
def _chroma_client(persist_dir: Path):
    try:
        import chromadb
    except ImportError as exc:
        raise RuntimeError("Install chromadb: pip install chromadb") from exc
    persist_dir.mkdir(parents=True, exist_ok=True)
    return chromadb.PersistentClient(path=str(persist_dir))
def load_collection(persist_dir: Path, collection_name: str = COLLECTION_NAME):
    client = _chroma_client(persist_dir)
    return client.get_or_create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
def _sanitize_metadata(meta: dict[str, Any]) -> dict[str, Any]:
    clean: dict[str, Any] = {}
    for key, value in meta.items():
        if value is None:
            clean[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean[key] = value
        else:
            clean[key] = str(value)
    return clean


### Viva explanation

Vector store is Chroma, cosine space. Similarity later is `1 - distance`.

### Likely viva question

Where do you actually use ChromaDB?

### Answer

`_chroma_client` / `load_collection` / `collection.add` / `collection.query` in `index.py` and `retriever.py`.

### MASTER CODE ID: M019

**Source file:** `V2/src/retrieval/index.py`
**Function/Class:** `build_knowledge_base`

### What this cell does

Extracts, chunks, embeds, `collection.add` in batches of 100, writes `index_manifest.json`.

### Libraries used

chromadb, sentence-transformers, PyMuPDF (via helpers).

### Inputs

Corpus docs + local PDF paths; chunk 900/150; BGE-small.

### Outputs

Manifest with docs_indexed/chunks; gold context explicitly not ingested.

### Why this matters

This populates Chroma. Do not run it in the viva (`REBUILD_KB=False`).

In [ ]:
# Copied from V2/src/retrieval/index.py
"""Build and load the persistent Chroma knowledge-base index."""
from __future__ import annotations
from datetime import datetime, timezone
import json
from pathlib import Path
from typing import Any
from src.retrieval.chunking import chunk_pages
from src.retrieval.embeddings import embed_texts
from src.retrieval.extract import extract_pdf_pages
from src.retrieval.pdf_fetch import CorpusDoc

def build_knowledge_base(
    docs: list[CorpusDoc],
    local_paths: dict[str, str],
    *,
    persist_dir: Path,
    documents_dir: Path,
    chunk_size: int = 900,
    chunk_overlap: int = 150,
    embedding_model: str = "BAAI/bge-small-en-v1.5",
    collection_name: str = COLLECTION_NAME,
    reset: bool = True,
) -> dict[str, Any]:
    """Extract, chunk, embed, and persist Chroma index from downloaded PDFs."""
    client = _chroma_client(persist_dir)
    if reset:
        try:
            client.delete_collection(collection_name)
        except Exception:
            pass
    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    all_texts: list[str] = []
    all_ids: list[str] = []
    all_metas: list[dict[str, Any]] = []
    docs_indexed = 0
    docs_empty = 0
    extract_failures: list[dict[str, str]] = []

    for doc in docs:
        local = local_paths.get(doc.doc_key)
        if not local:
            extract_failures.append({"doc_key": doc.doc_key, "error": "missing_local_pdf"})
            continue
        pdf_path = Path(local)
        try:
            pages = extract_pdf_pages(pdf_path)
        except Exception as exc:  # noqa: BLE001
            extract_failures.append({"doc_key": doc.doc_key, "error": str(exc)})
            continue
        if not pages:
            docs_empty += 1
            continue

        base_meta = {
            "doc_id": doc.doc_key,
            "split": doc.split,
            "file_name": doc.file_name,
            "repo_pdf_path": doc.repo_path,
            "role": doc.role,
            "company_symbol": doc.company_symbol,
            "company_name": doc.company_name,
            "report_year": doc.report_year,
            "context_id": doc.context_id,
            "question_id": doc.question_id,
            "source_type": "pdf",
        }
        chunks = chunk_pages(
            pages,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            base_metadata=base_meta,
        )
        if not chunks:
            docs_empty += 1
            continue

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc.doc_key}::chunk_{i:04d}"
            meta = _sanitize_metadata({**chunk["metadata"], "chunk_id": chunk_id})
            all_ids.append(chunk_id)
            all_texts.append(chunk["text"])
            all_metas.append(meta)
        docs_indexed += 1

    if not all_texts:
        raise RuntimeError("No chunks produced — cannot build knowledge base")

    embeddings = embed_texts(all_texts, model_name=embedding_model)

    # Add in batches to avoid oversized requests.
    batch_size = 100
    for start in range(0, len(all_ids), batch_size):
        end = start + batch_size
        collection.add(
            ids=all_ids[start:end],
            documents=all_texts[start:end],
            embeddings=embeddings[start:end],
            metadatas=all_metas[start:end],
        )

    manifest = {
        "phase": 6,
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "collection_name": collection_name,
        "persist_dir": str(persist_dir),
        "documents_dir": str(documents_dir),
        "embedding_model": embedding_model,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "vector_store": "chroma",
        "docs_requested": len(docs),
        "docs_indexed": docs_indexed,
        "docs_empty_text": docs_empty,
        "chunks": len(all_ids),
        "roles": {
            "test": sum(1 for d in docs if d.role == "test"),
            "calibration": sum(1 for d in docs if d.role == "calibration"),
            "distractor": sum(1 for d in docs if d.role == "distractor"),
        },
        "extract_failures": extract_failures,
        "note": (
            "Index is built from FinQA source page PDFs only. "
            "Gold context fields are not ingested as retrieval documents."
        ),
    }
    manifest_path = persist_dir / "index_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return manifest


### Viva explanation

The manifest note is the leakage control: source PDFs only.

### Likely viva question

Where is the vector database populated?

### Answer

`collection.add(...)` inside `build_knowledge_base()`.

### MASTER CODE ID: M020

**Source file:** `V2/src/retrieval/retriever.py`
**Function/Class:** `retrieve / RetrievedChunk / _distance_to_similarity`

### What this cell does

Embeds the question, `collection.query` top_k=4, converts Chroma cosine distance to similarity.

### Libraries used

chromadb via `load_collection`; BGE via `embed_texts`.

### Inputs

question, persist_dir, top_k=4, embedding model, collection name.

### Outputs

List of `RetrievedChunk` with text, score, file_name, provenance.

### Why this matters

Shared retrieval for all three architectures.

In [ ]:
# Copied from V2/src/retrieval/retriever.py
"""Query the persistent knowledge-base index."""
from __future__ import annotations
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
from src.retrieval.embeddings import embed_texts
from src.retrieval.index import COLLECTION_NAME, load_collection

@dataclass
class RetrievedChunk:
    chunk_id: str
    text: str
    score: float
    doc_id: str
    file_name: str
    split: str
    page: int | str
    company_symbol: str
    report_year: str
    role: str
    context_id: str
    source_type: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)
def _distance_to_similarity(distance: float | None) -> float:
    if distance is None:
        return 0.0
    # Chroma cosine space returns distance; convert to a bounded similarity-like score.
    return max(0.0, 1.0 - float(distance))
def retrieve(
    question: str,
    *,
    persist_dir: Path,
    top_k: int = 4,
    embedding_model: str = "BAAI/bge-small-en-v1.5",
    collection_name: str = COLLECTION_NAME,
) -> list[RetrievedChunk]:
    if not question or not question.strip():
        raise ValueError("question must be non-empty")

    collection = load_collection(persist_dir, collection_name=collection_name)
    query_vec = embed_texts([question], model_name=embedding_model)[0]
    result = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    documents = (result.get("documents") or [[]])[0]
    metadatas = (result.get("metadatas") or [[]])[0]
    distances = (result.get("distances") or [[]])[0]
    ids = (result.get("ids") or [[]])[0]

    chunks: list[RetrievedChunk] = []
    for i, text in enumerate(documents):
        meta = metadatas[i] if i < len(metadatas) else {}
        distance = distances[i] if i < len(distances) else None
        chunk_id = ids[i] if i < len(ids) else str(meta.get("chunk_id") or f"idx_{i}")
        chunks.append(
            RetrievedChunk(
                chunk_id=str(chunk_id),
                text=str(text or ""),
                score=_distance_to_similarity(distance),
                doc_id=str(meta.get("doc_id") or ""),
                file_name=str(meta.get("file_name") or ""),
                split=str(meta.get("split") or ""),
                page=meta.get("page", ""),
                company_symbol=str(meta.get("company_symbol") or ""),
                report_year=str(meta.get("report_year") or ""),
                role=str(meta.get("role") or ""),
                context_id=str(meta.get("context_id") or ""),
                source_type=str(meta.get("source_type") or ""),
            )
        )
    return chunks


### Viva explanation

Single-Agent, Multi-Agent and UQ all call this same `retrieve()`. Top-k is 4.

### Likely viva question

Where is retrieval performed, and where is top-k?

### Answer

`retrieve()` in `retriever.py`; `retrieval.top_k: 4` in `experiment.yaml`, passed as `top_k`.

### MASTER CODE ID: M021

**Source file:** `V2/scripts/build_index.py`
**Function/Class:** `main (Phase 6 CLI)`

### What this cell does

Wires CSV freezes → collect targets → download → `build_knowledge_base`. Default 50 distractors.

### Libraries used

argparse + retrieval modules.

### Inputs

`--distractors 50` (Phase 8 Colab also calls this).

### Outputs

Index + `results/config/phase6_index_manifest.json`.

### Why this matters

Historical KB build entrypoint. Labelled do-not-run for viva.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: scripts/build_index.py rebuilds the knowledge base
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: scripts/build_index.py rebuilds the knowledge base')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('# Copied from V2/scripts/build_index.py lines 23–98\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="Build FinQA source-PDF knowledge base")\n    parser.add_argument("--distractors", type=int, default=50, help="Extra train PDFs as distractors")\n    parser.add_argument("--skip-download", action="store_true", help="Use already-downloaded PDFs only")\n    parser.add_argument("--no-reset", action="store_true", help="Do not delete existing collection")\n    parser.add_argument("--demo-query", type=str, default="", help="Optional retrieval smoke query")\n    args = parser.parse_args()\n\n    config = load_experiment_config()\n    run_id = create_run_id("phase6")\n    setup_logging(\n        level="INFO",\n        log_dir=get_path(config, "results_logs"),\n        run_id=run_id,\n        console=True,\n        file=True,\n    )\n    log = get_logger(run_id=run_id, phase="phase6")\n\n    test_csv = get_path(config, "data_final") / "selected_140_questions.csv"\n    cal_csv = get_path(config, "data_calibration") / "calibration_questions.csv"\n    docs_dir = get_path(config, "kb_documents")\n    index_dir = get_path(config, "kb_index")\n    repo_id = str(config.get("dataset", "huggingface_id", default="G4KMU/t2-ragbench"))\n    subset = str(config.get("dataset", "subset", default="FinQA"))\n    emb = str(config.get("embeddings", "model", default="BAAI/bge-small-en-v1.5"))\n    chunk_size = int(config.get("retrieval", "chunk_size", default=900))\n    chunk_overlap = int(config.get("retrieval", "chunk_overlap", default=150))\n    top_k = int(config.get("retrieval", "top_k", default=4))\n    seed = int(config.get("execution", "random_seed", default=42))\n\n    log.info("Collecting corpus targets distractors=%s", args.distractors)\n    docs = collect_corpus_targets(\n        test_csv=test_csv,\n        calibration_csv=cal_csv,\n        distractor_count=args.distractors,\n        distractor_seed=seed,\n        dataset_id=repo_id,\n        subset=subset,\n    )\n    log.info("Corpus docs=%s", len(docs))\n\n    if args.skip_download:\n        local_paths = {}\n        for doc in docs:\n            path = docs_dir / doc.split / doc.file_name\n            if path.exists():\n                local_paths[doc.doc_key] = str(path)\n        download_stats = {\n            "requested": len(docs),\n            "downloaded": 0,\n            "skipped_existing": len(local_paths),\n            "failed": [],\n            "local_paths": local_paths,\n        }\n    else:\n        log.info("Downloading PDFs into %s", docs_dir)\n        download_stats = download_pdfs(docs, documents_dir=docs_dir, repo_id=repo_id)\n    log.info(\n        "Download done downloaded=%s skipped=%s failed=%s",\n        download_stats["downloaded"],\n        download_stats["skipped_existing"],\n        len(download_stats["failed"]),\n    )\n\n    log.info("Building index at %s", index_dir)\n    manifest = build_knowledge_base(\n        docs,\n        download_stats["local_paths"],\n        persist_dir=index_dir,\n        documents_dir=docs_dir,\n        chunk_size=chunk_size,\n        chunk_overlap=chunk_overlap,\n        embedding_model=emb,\n        reset=not args.no_reset,\n    )\n')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: scripts/build_index.py rebuilds the knowledge base)'
    )


### Viva explanation

I will not rebuild Chroma in the viva. The existing Phase 6 index is restored on Colab from Drive.

### Likely viva question

Which script built the knowledge base?

### Answer

`V2/scripts/build_index.py`.

## PHASE 7 — Qwen3-8B backend

Notebook: `V2/notebooks/colab_phase7_smoke.ipynb` (historical smoke).

### MASTER CODE ID: M022

**Source file:** `V2/src/models/llama_cpp_backend.py`
**Function/Class:** `LlamaCppBackend`

### What this cell does

Loads GGUF via llama.cpp, n_ctx=4096, n_gpu_layers=-1, generate() with temperature and max_tokens.

### Libraries used

llama_cpp (`Llama`, `Llama.from_pretrained`).

### Inputs

hf_repo `bartowski/Qwen_Qwen3-8B-GGUF`, filename `Qwen_Qwen3-8B-Q4_K_M.gguf`.

### Outputs

`GenerationResult` with text, backend=`llama_cpp`, quantisation=`Q4_K_M`.

### Why this matters

Official Colab inference path. Ollama is not this class.

In [ ]:
# Copied from V2/src/models/llama_cpp_backend.py
"""llama.cpp GGUF backend for Colab GPU (primary remote path)."""
from __future__ import annotations
from pathlib import Path
import time
from typing import Any
from src.models.types import GenerationResult

class LlamaCppBackend:
    name = "llama_cpp"

    def __init__(
        self,
        *,
        model_path: str | None = None,
        hf_repo_id: str = "bartowski/Qwen_Qwen3-8B-GGUF",
        gguf_filename: str = "Qwen_Qwen3-8B-Q4_K_M.gguf",
        quantisation: str = "Q4_K_M",
        n_ctx: int = 4096,
        n_gpu_layers: int = -1,
        model_name: str = "Qwen3-8B",
    ) -> None:
        self.model_path = model_path
        self.hf_repo_id = hf_repo_id
        self.gguf_filename = gguf_filename
        self.quantisation = quantisation
        self.n_ctx = n_ctx
        self.n_gpu_layers = n_gpu_layers
        self.model_name = model_name
        self._llm = None

    def is_available(self) -> bool:
        try:
            import llama_cpp  # noqa: F401
        except ImportError:
            return False
        if self.model_path and Path(self.model_path).exists():
            return True
        # Available as a backend implementation even if weights not downloaded yet.
        return True

    def _load(self) -> Any:
        if self._llm is not None:
            return self._llm
        from llama_cpp import Llama

        if self.model_path and Path(self.model_path).exists():
            self._llm = Llama(
                model_path=self.model_path,
                n_ctx=self.n_ctx,
                n_gpu_layers=self.n_gpu_layers,
                verbose=False,
            )
            return self._llm

        # Download from Hugging Face on first use (Colab-friendly).
        self._llm = Llama.from_pretrained(
            repo_id=self.hf_repo_id,
            filename=self.gguf_filename,
            n_ctx=self.n_ctx,
            n_gpu_layers=self.n_gpu_layers,
            verbose=False,
        )
        return self._llm

    def generate(
        self,
        prompt: str,
        *,
        temperature: float = 0.1,
        max_new_tokens: int = 512,
        top_p: float | None = None,
    ) -> GenerationResult:
        llm = self._load()
        start = time.perf_counter()
        kwargs: dict[str, Any] = {
            "prompt": prompt,
            "max_tokens": max_new_tokens,
            "temperature": temperature,
        }
        if top_p is not None:
            kwargs["top_p"] = top_p
        output = llm(**kwargs)
        latency = time.perf_counter() - start
        choice = (output.get("choices") or [{}])[0]
        text = str(choice.get("text") or "").strip()
        return GenerationResult(
            text=text,
            model=self.model_name,
            backend=self.name,
            quantisation=self.quantisation,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            latency_seconds=latency,
            prompt_chars=len(prompt),
            finish_reason=choice.get("finish_reason"),
            raw={"usage": output.get("usage")},
        )


### Viva explanation

Official generation is Qwen3-8B Q4_K_M through llama.cpp on Tesla T4. Ollama is local-dev only.

### Likely viva question

Where is the model loaded and called?

### Answer

`LlamaCppBackend._load` / `generate` in `llama_cpp_backend.py`.

### MASTER CODE ID: M023

**Source file:** `V2/src/models/factory.py`
**Function/Class:** `create_backend`

### What this cell does

Selects llama_cpp / transformers / ollama_dev / mock. Live demo forbids mock and Ollama.

### Libraries used

backend classes + `runtime_guard`.

### Inputs

`model` section of experiment.yaml (`backend: auto`).

### Outputs

An `LLMBackend` instance.

### Why this matters

Shows why Ollama is not the viva stack when `V2_FORBID_MOCK` is set.

In [ ]:
# Copied from V2/src/models/factory.py
"""Factory for selecting an LLM backend from experiment config."""
from __future__ import annotations
from typing import Any
from src.models.llama_cpp_backend import LlamaCppBackend
from src.models.mock_backend import MockBackend
from src.models.ollama_backend import OllamaBackend
from src.models.runtime_guard import LiveRuntimeError, assert_not_mock_backend, mock_forbidden
from src.models.transformers_backend import TransformersBackend
from src.models.types import LLMBackend

def create_backend(model_cfg: dict[str, Any] | None = None) -> LLMBackend:
    """Create a backend.

    Preferred Colab order when ``backend: auto``:
    1. llama_cpp (GGUF) if importable
    2. transformers if importable
    3. ollama_dev if a local model is present (dev only)
    """
    cfg = dict(model_cfg or {})
    if mock_forbidden():
        cfg["backend"] = "llama_cpp"
    backend = str(cfg.get("backend") or "auto").lower()
    assert_not_mock_backend(backend)

    if backend in {"mock", "test"}:
        if mock_forbidden():
            raise LiveRuntimeError("Mock backend is forbidden for the Colab live demo.")
        return MockBackend()

    if backend in {"ollama", "ollama_dev"}:
        if mock_forbidden():
            raise LiveRuntimeError("Local Ollama is forbidden for the Colab live demo. Use llama_cpp.")
        return OllamaBackend(model=str(cfg.get("ollama_model") or "qwen3:8b"))

    if backend in {"llama_cpp", "llamacpp", "gguf", "colab"}:
        return LlamaCppBackend(
            model_path=cfg.get("model_path"),
            hf_repo_id=str(cfg.get("hf_repo_id") or "bartowski/Qwen_Qwen3-8B-GGUF"),
            gguf_filename=str(cfg.get("gguf_filename") or "Qwen_Qwen3-8B-Q4_K_M.gguf"),
            quantisation=str(cfg.get("quantisation") or "Q4_K_M"),
            n_ctx=int(cfg.get("n_ctx") or 4096),
            n_gpu_layers=int(cfg.get("n_gpu_layers") if cfg.get("n_gpu_layers") is not None else -1),
            model_name=str(cfg.get("name") or "Qwen3-8B"),
        )

    if backend in {"transformers", "hf", "huggingface"}:
        return TransformersBackend(
            model_id=str(cfg.get("hf_model_id") or "Qwen/Qwen3-8B"),
            quantisation=str(cfg.get("quantisation") or "bitsandbytes-4bit"),
            load_in_4bit=bool(cfg.get("load_in_4bit", True)),
        )

    if backend == "auto":
        llama = LlamaCppBackend(
            model_path=cfg.get("model_path"),
            hf_repo_id=str(cfg.get("hf_repo_id") or "bartowski/Qwen_Qwen3-8B-GGUF"),
            gguf_filename=str(cfg.get("gguf_filename") or "Qwen_Qwen3-8B-Q4_K_M.gguf"),
            quantisation=str(cfg.get("quantisation") or "Q4_K_M"),
            model_name=str(cfg.get("name") or "Qwen3-8B"),
        )
        try:
            import llama_cpp  # noqa: F401

            return llama
        except ImportError:
            pass

        transformers = TransformersBackend(
            model_id=str(cfg.get("hf_model_id") or "Qwen/Qwen3-8B"),
        )
        if transformers.is_available():
            # Prefer transformers only when CUDA is present for 8B practicality.
            try:
                import torch

                if torch.cuda.is_available():
                    return transformers
            except Exception:
                pass

        ollama = OllamaBackend(model=str(cfg.get("ollama_model") or "qwen3:8b"))
        if ollama.is_available():
            return ollama

        raise RuntimeError(
            "No LLM backend available. On Colab install llama-cpp-python (CUDA) or "
            "transformers+bitsandbytes. For local smoke only, install/start Ollama with qwen3:8b."
        )

    raise ValueError(f"Unknown model backend: {backend}")


### Viva explanation

If the professor asks about Ollama: it exists as `OllamaBackend` for Mac smoke. Phase 21 locks llama_cpp.

### Likely viva question

Is Ollama the official backend?

### Answer

No. Official Colab path is `LlamaCppBackend`. Factory forbids Ollama when mock is forbidden.

### MASTER CODE ID: M024

**Source file:** `V2/notebooks/colab_phase7_smoke.ipynb`
**Source notebook:** `V2/notebooks/colab_phase7_smoke.ipynb`
**Original notebook cell:** Cell 9
**Function/Class:** `Phase 7 smoke command`

### What this cell does

Original Colab cell that ran `scripts/smoke_generate.py --backend llama_cpp`.

### Libraries used

llama_cpp on Colab GPU (via the script).

### Inputs

Colab T4 runtime.

### Outputs

`results/config/phase7_smoke_test.json`.

### Why this matters

Historical GPU smoke. Not the 420-case benchmark.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 7 smoke_generate
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 7 smoke_generate')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 7 smoke_generate)'
    )


### Viva explanation

Phase 7 only proved Qwen generates on T4. It is labelled historical engineering.

### Likely viva question

Did Phase 7 run the 420 cases?

### Answer

No. It smoked `scripts/smoke_generate.py`.

## PHASE 8 — Single-Agent RAG baseline

Notebook: `V2/notebooks/colab_phase8_smoke.ipynb`.

### MASTER CODE ID: M025

**Source file:** `V2/src/rag/schema.py`
**Function/Class:** `RAGCaseResult`

### What this cell does

Shared raw record for one architecture–question case (evidence, answer, verification, confidence, decision).

### Libraries used

dataclasses.

### Inputs

Filled by `run_single_agent` / `run_multi_agent` / `run_multi_agent_uq`.

### Outputs

`to_dict()` written to JSONL.

### Why this matters

All three architectures write the same schema so Phase 16 can score them uniformly.

In [ ]:
# Copied from V2/src/rag/schema.py
"""Common RAG case-result schema aligned with storage.raw_result_fields."""
from __future__ import annotations
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any

ARCHITECTURE_SINGLE_AGENT = "single_agent"
ARCHITECTURE_MULTI_AGENT = "multi_agent"
ARCHITECTURE_MULTI_AGENT_UQ = "multi_agent_uq"
@dataclass
class RAGCaseResult:
    """One architecture–question evaluation record (raw, before aggregation)."""

    run_id: str
    question_id: str
    architecture: str
    question: str
    retrieved_evidence: list[dict[str, Any]]
    retrieval_scores: list[float]
    answer: str
    reference_answer: str | None = None
    verification_result: Any = None  # Phase 9+
    confidence: float | None = None  # Phase 10+
    threshold: float | None = None  # Phase 10+
    decision: str = "ANSWER"  # baseline always answers; abstention is Phase 10
    latency_seconds: float = 0.0
    model: str | None = None
    model_version: str | None = None
    quantisation: str | None = None
    device: str | None = None
    gpu: str | None = None
    configuration: dict[str, Any] = field(default_factory=dict)
    random_seed: int | None = None
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    error: str | None = None
    # Extra provenance (allowed beyond minimal field list)
    retrieval_top_k: int | None = None
    backend: str | None = None
    prompt_chars: int | None = None
    case_key: str | None = None

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @property
    def case_id(self) -> str:
        return self.case_key or f"{self.architecture}:{self.question_id}"


### Viva explanation

Single-Agent sets `verification_result=None`, `confidence=None`, `decision='ANSWER'`.

### Likely viva question

What is stored per case?

### Answer

`RAGCaseResult` in `V2/src/rag/schema.py`.

### MASTER CODE ID: M026

**Source file:** `V2/src/rag/prompts.py`
**Function/Class:** `format_evidence / build_baseline_prompt`

### What this cell does

Builds the Single-Agent prompt: evidence block + question + ‘Final answer (once only)’.

### Libraries used

YAML prompts via `load_prompts_config`.

### Inputs

question + retrieved chunks; `config/prompts.yaml` baseline section.

### Outputs

One string prompt for Qwen.

### Why this matters

Evidence grounding is literally the retrieved chunks concatenated into the prompt.

In [ ]:
# Copied from V2/src/rag/prompts.py
"""Prompt construction for V2 RAG architectures."""
from __future__ import annotations
from typing import Any
from src.config import load_prompts_config
from src.retrieval.retriever import RetrievedChunk

DEFAULT_BASELINE_SYSTEM = (
    "Answer the financial question using only the evidence below. "
    "Write the final answer once, in one short sentence or one number with its unit. "
    "Do not repeat the answer. Do not repeat these instructions. Do not write reasoning. "
    "Distinguish the quantity the question asks for: final/ending/cumulative value is not "
    "the same as absolute change, and neither is the same as percentage change or ROI. "
    "If the question asks for ROI or percentage change, do not report the ending investment value. "
    "If the evidence does not contain the answer, write exactly: Evidence is insufficient."
)
DEFAULT_BASELINE_USER = """Evidence:
{evidence}

Question:
{question}

Final answer (once only):"""
def format_evidence(chunks: list[RetrievedChunk] | list[dict[str, Any]]) -> str:
    parts: list[str] = []
    for i, chunk in enumerate(chunks, start=1):
        if isinstance(chunk, dict):
            text = str(chunk.get("text") or "")
            file_name = str(chunk.get("file_name") or "")
            score = chunk.get("score")
        else:
            text = chunk.text
            file_name = chunk.file_name
            score = chunk.score
        header = f"[{i}] file={file_name} score={score:.4f}" if score is not None else f"[{i}] file={file_name}"
        parts.append(f"{header}\n{text.strip()}")
    return "\n\n".join(parts) if parts else "(no evidence retrieved)"
def build_baseline_prompt(
    question: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    baseline = cfg.get("baseline") or {}
    system = baseline.get("system") or DEFAULT_BASELINE_SYSTEM
    user_template = baseline.get("user_template") or DEFAULT_BASELINE_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"


### Viva explanation

The model is told to use only the evidence and to write the answer once. Fallback string is ‘Evidence is insufficient.’

### Likely viva question

Where are generation prompts built?

### Answer

`build_baseline_prompt()` and the multi-agent builders in `prompts.py`, templates in `prompts.yaml`.

### MASTER CODE ID: M027

**Source file:** `V2/src/rag/single_agent.py`
**Function/Class:** `run_single_agent`

### What this cell does

Baseline RAG: `retrieve` → `build_baseline_prompt` → `llm.generate` → always `decision='ANSWER'`.

### Libraries used

retriever, llama_cpp backend factory, prompts.

### Inputs

question, optional backend, config (top_k=4, collection `finqa_source_pdfs`).

### Outputs

`RAGCaseResult` with evidence, answer, no verification, no abstention.

### Why this matters

This is architecture 1 — the controlled baseline.

In [ ]:
# Copied from V2/src/rag/single_agent.py
"""Single-Agent RAG baseline (Phase 8)."""
from __future__ import annotations
import time
from pathlib import Path
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.prompts import build_baseline_prompt
from src.rag.schema import ARCHITECTURE_SINGLE_AGENT, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.retrieval.retriever import retrieve
from src.utils import create_run_id

def run_single_agent(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
) -> RAGCaseResult:
    """Run retrieve → prompt → generate for one question."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase8")
    architecture = ARCHITECTURE_SINGLE_AGENT
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    prompt_chars = 0

    try:
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]
        prompts_cfg = load_prompts_config()
        prompt = build_baseline_prompt(question, chunks, prompts_cfg=prompts_cfg)
        prompt_chars = len(prompt)
        gen = llm.generate(
            prompt,
            temperature=float(model_cfg.get("temperature") or 0.1),
            max_new_tokens=int(model_cfg.get("max_new_tokens") or 512),
            top_p=model_cfg.get("top_p"),
        )
        answer = clean_generated_answer(gen.text or "")
        # One retry if the backend returns empty text (seen with some local Ollama/Qwen3 runs).
        if not answer:
            gen = llm.generate(
                prompt,
                temperature=float(model_cfg.get("temperature") or 0.1),
                max_new_tokens=int(model_cfg.get("max_new_tokens") or 512),
                top_p=model_cfg.get("top_p"),
            )
            answer = clean_generated_answer(gen.text or "")
        model_name = gen.model
        quant = gen.quantisation
        backend_used = gen.backend
        if not answer:
            error = "Generation returned empty text after retry"
    except Exception as exc:  # noqa: BLE001 — capture into result record
        error = f"{type(exc).__name__}: {exc}"
        model_name = str(model_cfg.get("name") or "Qwen3-8B")
        quant = model_cfg.get("quantisation")
        backend_used = getattr(llm, "name", model_cfg.get("backend"))

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=answer,
        reference_answer=reference_answer,
        verification_result=None,
        confidence=None,
        threshold=None,
        decision="ANSWER",
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": model_cfg.get("temperature"),
            "max_new_tokens": model_cfg.get("max_new_tokens"),
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=prompt_chars,
        case_key=case_key,
    )


### Viva explanation

This is my baseline RAG architecture. The question is retrieved against Chroma, then Qwen generates. There is no verifier and no abstention gate; decision is hard-coded ANSWER.

### Likely viva question

Where is the Single-Agent architecture implemented?

### Answer

`run_single_agent()` in `V2/src/rag/single_agent.py`.

### MASTER CODE ID: M028

**Source file:** `V2/notebooks/colab_phase8_smoke.ipynb`
**Source notebook:** `V2/notebooks/colab_phase8_smoke.ipynb`
**Original notebook cell:** Cell 12
**Function/Class:** `Phase 8 smoke command`

### What this cell does

Original Colab cell: `scripts/smoke_single_agent.py --backend llama_cpp --limit 3`.

### Libraries used

llama_cpp (via script).

### Inputs

Existing Phase 6 index on Colab.

### Outputs

`phase8_smoke_test.json` (3 questions).

### Why this matters

Historical smoke, not 420.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 8 smoke_single_agent
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 8 smoke_single_agent')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('!PYTHONPATH=. python scripts/smoke_single_agent.py --backend llama_cpp --limit 3')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 8 smoke_single_agent)'
    )


### Viva explanation

Phase 8 Colab smoke used n=3. Official evaluation is Phase 15.

### Likely viva question

Is the Phase 8 notebook the official evaluation?

### Answer

No. Historical Single-Agent smoke only.

## PHASE 9 — Multi-Agent RAG

Notebook: `V2/notebooks/colab_phase9_smoke.ipynb`.

### MASTER CODE ID: M029

**Source file:** `V2/src/rag/prompts.py`
**Function/Class:** `build_multi_agent_draft_prompt / build_multi_agent_verification_prompt`

### What this cell does

Draft prompt (same evidence+question pattern) and verification prompt (evidence, question, draft → 0–1 score).

### Libraries used

YAML `multi_agent.generation` / `multi_agent.verification`.

### Inputs

question, draft answer, chunks.

### Outputs

Two prompt strings. Verifier must reply with one number 0.00–1.00.

### Why this matters

Verification is a separate LLM call, not a rewrite of the draft.

In [ ]:
# Copied from V2/src/rag/prompts.py
"""Prompt construction for V2 RAG architectures."""
from __future__ import annotations
from typing import Any
from src.config import load_prompts_config
from src.retrieval.retriever import RetrievedChunk

def build_multi_agent_draft_prompt(
    question: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    section = (cfg.get("multi_agent") or {}).get("generation") or {}
    system = section.get("system") or DEFAULT_MULTI_AGENT_DRAFT_SYSTEM
    user_template = section.get("user_template") or DEFAULT_MULTI_AGENT_DRAFT_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"
def build_multi_agent_verification_prompt(
    question: str,
    answer: str,
    chunks: list[RetrievedChunk] | list[dict[str, Any]],
    *,
    prompts_cfg: dict[str, Any] | None = None,
) -> str:
    cfg = prompts_cfg if prompts_cfg is not None else load_prompts_config()
    section = (cfg.get("multi_agent") or {}).get("verification") or {}
    system = section.get("system") or DEFAULT_MULTI_AGENT_VERIFY_SYSTEM
    user_template = section.get("user_template") or DEFAULT_MULTI_AGENT_VERIFY_USER
    user = user_template.format(
        evidence=format_evidence(chunks),
        question=question.strip(),
        answer=answer.strip(),
    )
    return f"{system.strip()}\n\n{user.strip()}"


### Viva explanation

The checker does not replace the answer. It scores support.

### Likely viva question

Where is the verification prompt?

### Answer

`build_multi_agent_verification_prompt()` and `prompts.yaml` `multi_agent.verification`.

### MASTER CODE ID: M030

**Source file:** `V2/src/rag/text_utils.py`
**Function/Class:** `token_overlap / average / parse_unit_score`

### What this cell does

Lexical overlap for verification; mean of scores; parse a 0–1 number from Qwen’s verifier/judge output.

### Libraries used

re.

### Inputs

Answer text vs evidence text, or raw LLM score text.

### Outputs

Float scores in 0–1.

### Why this matters

These helpers are the lexical half of verification and the parser for the LLM half.

In [ ]:
# Copied from V2/src/rag/text_utils.py
"""Lightweight text helpers for verification scoring and generation cleanup."""
from __future__ import annotations
import re
from typing import Iterable

def token_overlap(reference: str, candidate: str) -> float:
    reference_tokens = {t for t in re.findall(r"[a-zA-Z0-9]+", reference.lower()) if len(t) > 2}
    candidate_tokens = {t for t in re.findall(r"[a-zA-Z0-9]+", candidate.lower()) if len(t) > 2}
    if not reference_tokens or not candidate_tokens:
        return 0.0
    return len(reference_tokens & candidate_tokens) / max(1, len(reference_tokens))
def average(values: Iterable[float]) -> float:
    items = list(values)
    return sum(items) / len(items) if items else 0.0
def parse_unit_score(text: str) -> float | None:
    """Extract a 0–1 score from model output.

    Ignores instruction echoes such as "between 0 and 1" so a repeated
    prompt cannot become a contradictory 0.0 support score.
    """
    raw = clean_generated_answer(text or "")
    if not raw:
        return None
    cleaned = _INSTRUCTION_ECHO.sub(" ", raw)
    decimals = [float(m) for m in re.findall(r"\b(0\.\d+|1\.0+|1)\b", cleaned)]
    if decimals:
        value = decimals[-1]
        return max(0.0, min(1.0, value))
    match = re.search(r"(\d+(?:\.\d+)?)", cleaned)
    if not match:
        return None
    try:
        value = float(match.group(1))
    except ValueError:
        return None
    if value > 1.0 and value <= 100.0:
        value = value / 100.0
    if value in {0.0, 1.0} and _INSTRUCTION_ECHO.search(raw):
        return None
    return max(0.0, min(1.0, value))


### Viva explanation

Lexical score is token Jaccard-style overlap on tokens longer than 2 characters.

### Likely viva question

How is lexical overlap computed?

### Answer

`token_overlap()` in `V2/src/rag/text_utils.py`.

### MASTER CODE ID: M031

**Source file:** `V2/src/rag/verification.py`
**Function/Class:** `compute_verification_result`

### What this cell does

Lexical overlap + LLM support score, averaged; status VERIFIED if score ≥ 0.50, else WEAK_EVIDENCE. No abstention.

### Libraries used

LLM backend generate; text_utils.

### Inputs

question, draft answer, chunks, llm, verification_threshold=0.5.

### Outputs

Dict with verification_score, lexical_score, llm_score, status, rationale.

### Why this matters

This is the Multi-Agent checker. It does not change `answer`.

In [ ]:
# Copied from V2/src/rag/verification.py
"""Evidence-grounded verification for Multi-Agent RAG (Phase 9)."""
from __future__ import annotations
from typing import Any
from src.models.types import LLMBackend
from src.rag.prompts import build_multi_agent_verification_prompt, format_evidence
from src.rag.text_utils import average, build_verification_rationale, parse_unit_score, token_overlap
from src.retrieval.retriever import RetrievedChunk

def compute_verification_result(
    question: str,
    answer: str,
    chunks: list[RetrievedChunk],
    llm: LLMBackend,
    *,
    prompts_cfg: dict[str, Any] | None = None,
    verification_threshold: float = 0.5,
    temperature: float = 0.0,
    max_new_tokens: int = 32,
) -> dict[str, Any]:
    """Lexical overlap + LLM support score (no abstention logic here)."""
    evidence_text = format_evidence(chunks)
    lexical_score = token_overlap(answer, evidence_text)
    llm_score: float | None = None

    if answer.strip():
        prompt = build_multi_agent_verification_prompt(
            question,
            answer,
            chunks,
            prompts_cfg=prompts_cfg,
        )
        gen = llm.generate(
            prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
        )
        llm_score = parse_unit_score(gen.text or "")

    if llm_score is not None:
        verification_score = average([lexical_score, llm_score])
    else:
        verification_score = lexical_score

    status = "VERIFIED" if verification_score >= verification_threshold else "WEAK_EVIDENCE"
    rationale = build_verification_rationale(
        status=status,
        verification_score=verification_score,
        lexical_score=lexical_score,
        llm_score=llm_score,
        verification_threshold=verification_threshold,
    )
    return {
        "verification_score": verification_score,
        "lexical_score": lexical_score,
        "llm_score": llm_score,
        "verification_threshold": verification_threshold,
        "status": status,
        "rationale": rationale,
    }


### Viva explanation

Verification is informational on the Multi-Agent path. The architecture still returns the draft as the answer and decision stays ANSWER.

### Likely viva question

Does verification rewrite the answer or abstain?

### Answer

No. `compute_verification_result` only returns scores/status. Abstention is Phase 10.

### MASTER CODE ID: M032

**Source file:** `V2/src/rag/multi_agent.py`
**Function/Class:** `run_multi_agent / _generate_with_retry`

### What this cell does

Retrieve → draft generate → `compute_verification_result`. `decision='ANSWER'`. Confidence field stores verification_score only.

### Libraries used

retriever, prompts, verification, backend factory.

### Inputs

Same shared KB/top_k as Single-Agent.

### Outputs

`RAGCaseResult` with `verification_result` populated.

### Why this matters

Architecture 2. Independent of architecture 1 — not a chain of stored answers.

In [ ]:
# Copied from V2/src/rag/multi_agent.py
"""Multi-Agent RAG (Phase 9): retrieve → draft → verify."""
from __future__ import annotations
import time
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import GenerationResult, LLMBackend
from src.rag.prompts import build_multi_agent_draft_prompt, build_multi_agent_verification_prompt
from src.rag.schema import ARCHITECTURE_MULTI_AGENT, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.rag.verification import compute_verification_result
from src.retrieval.retriever import retrieve
from src.utils import create_run_id

def _generate_with_retry(
    llm: LLMBackend,
    prompt: str,
    *,
    temperature: float,
    max_new_tokens: int,
    top_p: float | None,
) -> GenerationResult:
    gen = llm.generate(
        prompt,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        top_p=top_p,
    )
    if not (gen.text or "").strip():
        gen = llm.generate(
            prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
    return gen
def run_multi_agent(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
) -> RAGCaseResult:
    """Run retrieve → draft → verification for one question."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")
    rag_cfg = cfg.section("rag")
    multi_cfg = dict(rag_cfg.get("architectures", {}).get("multi_agent") or {})

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase9")
    architecture = ARCHITECTURE_MULTI_AGENT
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")
    verify_threshold = float(multi_cfg.get("verification_threshold") or 0.5)
    temperature = float(model_cfg.get("temperature") or 0.1)
    max_new_tokens = int(model_cfg.get("max_new_tokens") or 512)
    top_p = model_cfg.get("top_p")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    draft_prompt_chars = 0
    verify_prompt_chars = 0
    verification_result: dict[str, Any] | None = None
    model_name = str(model_cfg.get("name") or "Qwen3-8B")
    quant: Any = model_cfg.get("quantisation")
    backend_used: Any = getattr(llm, "name", model_cfg.get("backend"))

    try:
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]

        prompts_cfg = load_prompts_config()
        draft_prompt = build_multi_agent_draft_prompt(question, chunks, prompts_cfg=prompts_cfg)
        draft_prompt_chars = len(draft_prompt)
        draft_gen = _generate_with_retry(
            llm,
            draft_prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
        answer = clean_generated_answer(draft_gen.text or "")
        model_name = draft_gen.model
        quant = draft_gen.quantisation
        backend_used = draft_gen.backend

        if not answer:
            error = "Draft generation returned empty text after retry"
        else:
            verify_prompt = build_multi_agent_verification_prompt(
                question,
                answer,
                chunks,
                prompts_cfg=prompts_cfg,
            )
            verify_prompt_chars = len(verify_prompt)
            verification_result = compute_verification_result(
                question,
                answer,
                chunks,
                llm,
                prompts_cfg=prompts_cfg,
                verification_threshold=verify_threshold,
            )
    except Exception as exc:  # noqa: BLE001
        error = f"{type(exc).__name__}: {exc}"

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None
    confidence = (
        float(verification_result["verification_score"])
        if verification_result is not None
        else None
    )

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=answer,
        reference_answer=reference_answer,
        verification_result=verification_result,
        confidence=confidence,
        threshold=None,
        decision="ANSWER",
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "verification_threshold": verify_threshold,
            "draft_prompt_chars": draft_prompt_chars,
            "verify_prompt_chars": verify_prompt_chars,
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=draft_prompt_chars,
        case_key=case_key,
    )


### Viva explanation

Multi-Agent is sequential Python: retrieve, draft, verify. There is no LangGraph in this repository.

### Likely viva question

Where is Multi-Agent RAG implemented? Did you use LangGraph?

### Answer

`run_multi_agent()` in `multi_agent.py`. LangGraph is not imported anywhere under V2.

### MASTER CODE ID: M033

**Source file:** `V2/notebooks/colab_phase9_smoke.ipynb`
**Source notebook:** `V2/notebooks/colab_phase9_smoke.ipynb`
**Original notebook cell:** Cell 12
**Function/Class:** `Phase 9 smoke command`

### What this cell does

Original Colab cell: `scripts/smoke_multi_agent.py --backend llama_cpp --limit 3`.

### Libraries used

llama_cpp via script.

### Inputs

Restored Phase 6 index.

### Outputs

`phase9_smoke_test.json`.

### Why this matters

Historical n=3 smoke.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 9 smoke_multi_agent
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 9 smoke_multi_agent')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('!PYTHONPATH=. python scripts/smoke_multi_agent.py --backend llama_cpp --limit 3')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 9 smoke_multi_agent)'
    )


### Viva explanation

Phase 9 smoke is not the official 420.

### Likely viva question

Official Multi-Agent evaluation notebook?

### Answer

No. Official comparison is Phase 15. This cell is smoke.

## PHASE 10 — Multi-Agent RAG + UQ / abstention

Notebook: `V2/notebooks/colab_phase10_smoke.ipynb`.

### MASTER CODE ID: M034

**Source file:** `V2/src/rag/uncertainty.py`
**Function/Class:** `compute_retrieval_score / compute_combined_confidence / apply_abstention_decision`

### What this cell does

Retrieval score = mean of top-k similarities. Confidence = mean(retrieval, verification). Gate: confidence ≥ T → ANSWER else ABSTAIN.

### Libraries used

`average()` from text_utils.

### Inputs

retrieval_scores list, verification_score, threshold, draft_answer.

### Outputs

confidence dict; `(final_answer, decision)` where decision is ANSWER or ABSTAIN.

### Why this matters

This is the operational UQ rule. Not ECE/Brier. Method name is `mean_retrieval_verification`.

In [ ]:
# Copied from V2/src/rag/uncertainty.py
"""Uncertainty quantification and abstention for Multi-Agent + UQ (Phase 10)."""
from __future__ import annotations
from typing import Any
from src.rag.text_utils import average

def compute_retrieval_score(retrieval_scores: list[float]) -> float:
    """Aggregate top-k retrieval similarities into one signal (mean)."""
    if not retrieval_scores:
        return 0.0
    return average(retrieval_scores)
def compute_combined_confidence(
    retrieval_score: float,
    verification_score: float,
    *,
    method: str = "mean_retrieval_verification",
) -> dict[str, Any]:
    """Combine retrieval and verification into a single confidence score."""
    if method != "mean_retrieval_verification":
        raise ValueError(f"Unsupported UQ method: {method}")
    confidence = average([retrieval_score, verification_score])
    return {
        "method": method,
        "retrieval_score": retrieval_score,
        "verification_score": verification_score,
        "confidence": confidence,
    }
def apply_abstention_decision(
    *,
    draft_answer: str,
    confidence: float,
    threshold: float,
    abstention_message: str,
) -> tuple[str, str]:
    """Return (final_answer, decision) where decision is ANSWER or ABSTAIN."""
    if confidence >= threshold:
        return draft_answer, "ANSWER"
    return abstention_message, "ABSTAIN"


### Viva explanation

Confidence is the average of retrieval score and verification score. At the locked T=0.65, below T I abstain. It is a decision score, not a calibrated probability.

### Likely viva question

How is confidence calculated, and what is the gate?

### Answer

`compute_combined_confidence`: mean of the two scores. `apply_abstention_decision`: ≥ threshold ANSWER, else ABSTAIN.

### MASTER CODE ID: M035

**Source file:** `V2/src/rag/multi_agent_uq.py`
**Function/Class:** `run_multi_agent_uq / _resolve_threshold`

### What this cell does

Retrieve → draft → verify → combined confidence → abstention. Threshold from override, yaml lock, or smoke 0.55.

### Libraries used

multi_agent retry, verification, uncertainty, retriever.

### Inputs

question; threshold override used by Phase 13+ lock.

### Outputs

`RAGCaseResult` with `decision` ANSWER/ABSTAIN, `configuration.draft_answer` preserved.

### Why this matters

Architecture 3. Draft is kept so Phase 16 can score the claim even when displayed text is the abstention template.

In [ ]:
# Copied from V2/src/rag/multi_agent_uq.py
"""Multi-Agent RAG + UQ / abstention (Phase 10): retrieve → draft → verify → confidence gate."""
from __future__ import annotations
import time
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, load_prompts_config
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.multi_agent import _generate_with_retry
from src.rag.prompts import build_multi_agent_draft_prompt, build_multi_agent_verification_prompt
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.rag.text_utils import clean_generated_answer
from src.rag.uncertainty import (
    apply_abstention_decision,
    compute_combined_confidence,
    compute_retrieval_score,
)
from src.rag.verification import compute_verification_result
from src.retrieval.retriever import retrieve
from src.utils import create_run_id

def _resolve_threshold(cfg: ExperimentConfig, threshold_override: float | None) -> float:
    if threshold_override is not None:
        return float(threshold_override)
    uq_cfg = cfg.section("uncertainty")
    locked = uq_cfg.get("confidence_threshold")
    if locked is not None:
        return float(locked)
    smoke = uq_cfg.get("smoke_threshold")
    if smoke is not None:
        return float(smoke)
    return 0.55
def run_multi_agent_uq(
    question: str,
    *,
    question_id: str = "adhoc",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
    threshold: float | None = None,
) -> RAGCaseResult:
    """Run retrieve → draft → verify → combined confidence → abstention gate."""
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    retrieval_cfg = cfg.section("retrieval")
    embeddings_cfg = cfg.section("embeddings")
    execution_cfg = cfg.section("execution")
    rag_cfg = cfg.section("rag")
    uq_cfg = cfg.section("uncertainty")
    multi_cfg = dict(rag_cfg.get("architectures", {}).get("multi_agent") or {})
    prompts_cfg = load_prompts_config()
    uq_prompts = dict(prompts_cfg.get("uncertainty") or {})

    if backend_name:
        model_cfg["backend"] = backend_name

    from src.config import project_root

    rid = run_id or create_run_id("phase10")
    architecture = ARCHITECTURE_MULTI_AGENT_UQ
    case_key = f"{architecture}:{question_id}"

    top_k = int(retrieval_cfg.get("top_k") or 4)
    persist_dir = get_path(cfg, "kb_index")
    embed_model = str(embeddings_cfg.get("model") or "BAAI/bge-small-en-v1.5")
    collection = str(retrieval_cfg.get("collection_name") or "finqa_source_pdfs")
    verify_threshold = float(multi_cfg.get("verification_threshold") or 0.5)
    confidence_threshold = _resolve_threshold(cfg, threshold)
    uq_method = str(uq_cfg.get("method") or "mean_retrieval_verification")
    abstention_message = str(
        uq_prompts.get("abstention_message")
        or uq_cfg.get("abstention_message")
        or "I cannot answer reliably because supporting evidence is insufficient."
    )
    temperature = float(model_cfg.get("temperature") or 0.1)
    max_new_tokens = int(model_cfg.get("max_new_tokens") or 512)
    top_p = model_cfg.get("top_p")

    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )

    llm = backend or create_backend(model_cfg)
    start = time.perf_counter()
    error: str | None = None
    draft_answer = ""
    final_answer = ""
    chunks_dicts: list[dict[str, Any]] = []
    scores: list[float] = []
    draft_prompt_chars = 0
    verify_prompt_chars = 0
    verification_result: dict[str, Any] | None = None
    uncertainty_result: dict[str, Any] | None = None
    decision = "ANSWER"
    confidence: float | None = None
    model_name = str(model_cfg.get("name") or "Qwen3-8B")
    quant: Any = model_cfg.get("quantisation")
    backend_used: Any = getattr(llm, "name", model_cfg.get("backend"))

    try:
        chunks = retrieve(
            question,
            persist_dir=persist_dir,
            top_k=top_k,
            embedding_model=embed_model,
            collection_name=collection,
        )
        chunks_dicts = [c.to_dict() for c in chunks]
        scores = [float(c.score) for c in chunks]

        draft_prompt = build_multi_agent_draft_prompt(question, chunks, prompts_cfg=prompts_cfg)
        draft_prompt_chars = len(draft_prompt)
        draft_gen = _generate_with_retry(
            llm,
            draft_prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            top_p=top_p,
        )
        draft_answer = clean_generated_answer(draft_gen.text or "")
        model_name = draft_gen.model
        quant = draft_gen.quantisation
        backend_used = draft_gen.backend

        if not draft_answer:
            error = "Draft generation returned empty text after retry"
        else:
            verify_prompt = build_multi_agent_verification_prompt(
                question,
                draft_answer,
                chunks,
                prompts_cfg=prompts_cfg,
            )
            verify_prompt_chars = len(verify_prompt)
            verification_result = compute_verification_result(
                question,
                draft_answer,
                chunks,
                llm,
                prompts_cfg=prompts_cfg,
                verification_threshold=verify_threshold,
            )
            retrieval_score = compute_retrieval_score(scores)
            verify_score = float(verification_result["verification_score"])
            uncertainty_result = compute_combined_confidence(
                retrieval_score,
                verify_score,
                method=uq_method,
            )
            confidence = float(uncertainty_result["confidence"])
            final_answer, decision = apply_abstention_decision(
                draft_answer=draft_answer,
                confidence=confidence,
                threshold=confidence_threshold,
                abstention_message=abstention_message,
            )
    except Exception as exc:  # noqa: BLE001
        error = f"{type(exc).__name__}: {exc}"

    latency = time.perf_counter() - start
    gpu_info = (fp or {}).get("gpu") or {}
    gpu_name = gpu_info.get("name") if gpu_info.get("available") else None

    return RAGCaseResult(
        run_id=rid,
        question_id=str(question_id),
        architecture=architecture,
        question=question,
        retrieved_evidence=chunks_dicts,
        retrieval_scores=scores,
        answer=final_answer if not error else "",
        reference_answer=reference_answer,
        verification_result=verification_result,
        confidence=confidence,
        threshold=confidence_threshold,
        decision=decision,
        latency_seconds=latency,
        model=model_name,
        model_version=None,
        quantisation=str(quant) if quant is not None else None,
        device=(fp or {}).get("device"),
        gpu=gpu_name,
        configuration={
            "top_k": top_k,
            "embedding_model": embed_model,
            "collection_name": collection,
            "persist_dir": str(persist_dir),
            "hf_repo_id": model_cfg.get("hf_repo_id"),
            "gguf_filename": model_cfg.get("gguf_filename"),
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "verification_threshold": verify_threshold,
            "draft_prompt_chars": draft_prompt_chars,
            "verify_prompt_chars": verify_prompt_chars,
            "draft_answer": draft_answer or None,
            "uncertainty_result": uncertainty_result,
            "uq_method": uq_method,
            "threshold_source": (
                "override"
                if threshold is not None
                else ("locked" if uq_cfg.get("confidence_threshold") is not None else "smoke")
            ),
        },
        random_seed=execution_cfg.get("random_seed"),
        error=error,
        retrieval_top_k=top_k,
        backend=str(backend_used) if backend_used else None,
        prompt_chars=draft_prompt_chars,
        case_key=case_key,
    )


### Viva explanation

UQ reuses the same retrieve/draft/verify steps, then applies the confidence gate. Architectures are independent: I do not feed Single-Agent output into UQ.

### Likely viva question

Where is Multi-Agent + UQ implemented?

### Answer

`run_multi_agent_uq()` in `V2/src/rag/multi_agent_uq.py`.

### MASTER CODE ID: M036

**Source file:** `V2/notebooks/colab_phase10_smoke.ipynb`
**Source notebook:** `V2/notebooks/colab_phase10_smoke.ipynb`
**Original notebook cell:** Cell 12
**Function/Class:** `Phase 10 smoke command`

### What this cell does

Original Colab cell: `scripts/smoke_multi_agent_uq.py --backend llama_cpp --limit 3`.

### Libraries used

llama_cpp via script.

### Inputs

Smoke threshold 0.55 NOT LOCKED.

### Outputs

`phase10_smoke_test.json`.

### Why this matters

Historical. Official T is 0.65 from Phase 13, not this smoke 0.55.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 10 smoke_multi_agent_uq
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 10 smoke_multi_agent_uq')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('!PYTHONPATH=. python scripts/smoke_multi_agent_uq.py --backend llama_cpp --limit 3')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 10 smoke_multi_agent_uq)'
    )


### Viva explanation

If a smoke JSON shows 0.55, that is NOT the locked research threshold.

### Likely viva question

Is 0.55 the dissertation threshold?

### Answer

No. 0.55 is smoke/pilot. Locked T=0.65 is Phase 13 `threshold.lock.json`.

## PHASE 11 — Streamlit live artefact

Notebook: `V2/notebooks/colab_phase11_live.ipynb` (historical). Canonical launcher is Phase 21.

### MASTER CODE ID: M037

**Source file:** `V2/src/rag/live.py`
**Function/Class:** `run_live_comparison`

### What this cell does

Runs the three `run_*` functions independently on one question with locked T. `used_precomputed_benchmark_lookup=False`.

### Libraries used

single_agent, multi_agent, multi_agent_uq, lock file.

### Inputs

fresh or frozen question text; shared backend instance.

### Outputs

`LiveComparison` with three `RAGCaseResult`s.

### Why this matters

The artefact executes pipelines. It does not look up Phase 15 JSONL answers.

In [ ]:
# Copied from V2/src/rag/live.py
"""Live comparison runner: three independent RAG architectures on one question."""
from __future__ import annotations
import csv
import hashlib
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.runtime_guard import LiveRuntimeError, mock_forbidden
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.utils import create_run_id

def run_live_comparison(
    question: str,
    *,
    question_id: str | None = None,
    question_source: str = "fresh",
    reference_answer: str | None = None,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    backend_name: str | None = None,
    run_id: str | None = None,
    fingerprint: dict[str, Any] | None = None,
    threshold: float | None = None,
) -> LiveComparison:
    """Run the three architectures independently on the same original question.

    Always applies locked T from threshold.lock.json. The ``threshold`` argument is
    ignored so the live artefact cannot silently fall back to smoke 0.55.
    """
    cfg = config or load_experiment_config()
    model_cfg = dict(cfg.section("model"))
    if mock_forbidden():
        backend_name = "llama_cpp"
        model_cfg["backend"] = "llama_cpp"
        if backend is not None and str(getattr(backend, "name", "")).lower() in {"mock", "test"}:
            raise LiveRuntimeError(
                "Mock backend is forbidden for this live demo. Use llama_cpp on Colab T4."
            )
    elif backend_name:
        model_cfg["backend"] = backend_name

    locked_t = resolve_live_locked_threshold()
    _ = threshold
    rid = run_id or create_run_id("phase20")
    qid = question_id or make_fresh_question_id(question)
    fp = fingerprint or collect_fingerprint(
        model_config=model_cfg,
        project_root=str(project_root()),
    )
    llm = backend or create_backend(model_cfg)
    backend_used = str(getattr(llm, "name", model_cfg.get("backend")))
    if mock_forbidden() and backend_used.lower() in {"mock", "test"}:
        raise LiveRuntimeError(
            "Mock backend is forbidden for this live demo. Use llama_cpp on Colab T4."
        )

    comparison = LiveComparison(
        run_id=rid,
        question=question,
        question_id=qid,
        question_source=question_source,
        backend=backend_used,
        fingerprint=fp,
        used_precomputed_benchmark_lookup=False,
        locked_threshold=locked_t,
    )

    runners = (
        (ARCHITECTURE_SINGLE_AGENT, run_single_agent, {}),
        (ARCHITECTURE_MULTI_AGENT, run_multi_agent, {}),
        (ARCHITECTURE_MULTI_AGENT_UQ, run_multi_agent_uq, {"threshold": locked_t}),
    )

    try:
        for architecture, runner, extra in runners:
            kwargs: dict[str, Any] = {
                "question_id": qid,
                "reference_answer": reference_answer,
                "config": cfg,
                "backend": llm,
                "backend_name": backend_name,
                "run_id": rid,
                "fingerprint": fp,
            }
            kwargs.update(extra)
            case = normalize_live_case(runner(question, **kwargs))
            if architecture == ARCHITECTURE_MULTI_AGENT_UQ:
                case = annotate_live_uq_lock(case, locked_t)
            comparison.results[architecture] = case
    except Exception as exc:  # noqa: BLE001
        comparison.error = f"{type(exc).__name__}: {exc}"

    return comparison


### Viva explanation

Live demo calls the same Phase 8–10 functions. It is not a results browser pretending to generate.

### Likely viva question

Does Streamlit look up benchmark answers?

### Answer

No. `run_live_comparison` sets `used_precomputed_benchmark_lookup=False` and calls the three runners.

### MASTER CODE ID: M038

**Source file:** `V2/src/rag/live.py`
**Function/Class:** `normalize_live_case`

### What this cell does

Live-layer only: empty evidence/generation → ERROR/UNAVAILABLE; clears fabricated answers. Does not change Phase 8–10 modules.

### Libraries used

RAGCaseResult mutation.

### Inputs

A pipeline result.

### Outputs

Sanitised result for UI.

### Why this matters

Stops the UI showing ANSWER when retrieval failed.

In [ ]:
# Copied from V2/src/rag/live.py
"""Live comparison runner: three independent RAG architectures on one question."""
from __future__ import annotations
import csv
import hashlib
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.runtime_guard import LiveRuntimeError, mock_forbidden
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.utils import create_run_id

def live_run_failed(result: RAGCaseResult) -> bool:
    """True when the live artefact must not treat the case as a successful RAG run."""
    if result.error:
        return True
    if not result.retrieved_evidence:
        return True
    if not (result.answer or "").strip():
        return True
    return False
def normalize_live_case(result: RAGCaseResult) -> RAGCaseResult:
    """Live-app only: map failed retrieval/generation to ERROR/UNAVAILABLE.

    Does not change the Phase 8–10 pipeline modules. Clears fabricated answers
    and confidence when evidence is missing or generation failed.
    """
    if result.error:
        result.decision = DECISION_ERROR
        result.answer = ""
        result.confidence = None
        result.verification_result = None
        if result.configuration:
            result.configuration = dict(result.configuration)
            result.configuration.pop("uncertainty_result", None)
            result.configuration.pop("draft_answer", None)
        return result

    if not result.retrieved_evidence:
        result.decision = DECISION_UNAVAILABLE
        result.error = "No evidence retrieved; not a successful RAG run."
        result.answer = ""
        result.confidence = None
        result.verification_result = None
        if result.configuration:
            result.configuration = dict(result.configuration)
            result.configuration.pop("uncertainty_result", None)
            result.configuration.pop("draft_answer", None)
        return result

    if not (result.answer or "").strip():
        result.decision = DECISION_UNAVAILABLE
        result.error = "Generation returned empty text; not a successful RAG run."
        result.confidence = None
        result.verification_result = None
        return result

    if result.architecture == ARCHITECTURE_MULTI_AGENT_UQ:
        uq = (result.configuration or {}).get("uncertainty_result") or {}
        if uq.get("confidence") is not None:
            result.confidence = float(uq["confidence"])
        elif result.confidence is None:
            result.decision = DECISION_UNAVAILABLE
            result.error = "UQ confidence could not be calculated; not a valid zero-confidence result."
            result.confidence = None

    return result


### Viva explanation

If retrieval fails, I do not display a fake answer. That mapping is live-layer, not a fourth architecture.

### Likely viva question

What happens if retrieval returns nothing?

### Answer

`normalize_live_case` sets UNAVAILABLE, clears answer and confidence.

### MASTER CODE ID: M039

**Source file:** `V2/app/streamlit_app.py`
**Function/Class:** `render_architecture / render_evidence`

### What this cell does

Renders decision, answer, confidence, threshold, verification dict, UQ dict, evidence expanders.

### Libraries used

streamlit.

### Inputs

`RAGCaseResult`.

### Outputs

UI widgets. Uses markdown not `st.metric` for 0–1 confidence (Streamlit rounding bug).

### Why this matters

This is what the examiner sees for verification/confidence/abstention.

In [ ]:
# Copied from V2/app/streamlit_app.py
"""V2 live artefact: run all three RAG architectures on a fresh or frozen question."""
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
import streamlit as st
from app.benchmark_ui import render_benchmark_questions_page, render_benchmark_results_page
from src.config import get_path, load_experiment_config, project_root
from src.rag.benchmark_catalogue import (
    apply_catalogue_prefill_to_live_input,
    apply_pending_app_page,
)
from src.models.factory import create_backend
from src.models.runtime_guard import (
    LiveRuntimeError,
    live_demo_locked,
    verify_live_llama_cpp_runtime,
)
from src.rag.live import (
    ARCHITECTURE_LABELS,
    FRESH_KB_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION_ID,
    LIVE_ARCHITECTURES,
    LIVE_FAILURE_DECISIONS,
    format_confidence_display,
    format_optional,
    format_threshold_display,
    load_frozen_questions,
    resolve_displayed_confidence,
    resolve_live_locked_threshold,
    run_live_comparison,
    uq_ui_confidence_overlay,
)
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.retrieval.index import COLLECTION_NAME
from src.retrieval.preflight import IndexPreflightError, validate_index_preflight
from src.utils import create_run_id

def render_evidence(chunks: list[dict]) -> None:
    if not chunks:
        st.caption("No evidence retrieved.")
        return
    for idx, chunk in enumerate(chunks, start=1):
        score = chunk.get("score")
        title = f"[{idx}] {chunk.get('file_name') or chunk.get('doc_id') or 'chunk'}"
        if score is not None:
            title += f" · score {float(score):.4f}"
        with st.expander(title, expanded=idx == 1):
            st.markdown(
                f"**chunk_id:** `{chunk.get('chunk_id')}`  \n"
                f"**company:** {chunk.get('company_symbol')} · **year:** {chunk.get('report_year')}  \n"
                f"**role:** {chunk.get('role')} · **split:** {chunk.get('split')}"
            )
            st.write(chunk.get("text") or "")
def render_architecture(result: RAGCaseResult) -> None:
    label = ARCHITECTURE_LABELS.get(result.architecture, result.architecture)
    st.subheader(label)
    st.caption(f"`{result.architecture}` · `{result.case_key}`")

    failed = result.decision in LIVE_FAILURE_DECISIONS or bool(result.error) or not result.retrieved_evidence
    if failed:
        st.error(f"Status: {result.decision if result.decision in LIVE_FAILURE_DECISIONS else 'ERROR / UNAVAILABLE'}")
        st.error(result.error or "Retrieval or generation failed; this is not a successful RAG run.")
        st.markdown("**Generated answer**")
        st.caption("No answer (run failed). Nothing was fabricated.")
        col_a, col_b, col_c, col_d = st.columns(4)
        col_a.metric("Confidence", "n/a")
        col_b.markdown("**Threshold**")
        col_b.write(format_threshold_display(result))
        col_c.metric("Latency (s)", format_optional(result.latency_seconds))
        col_d.metric("Evidence chunks", str(len(result.retrieved_evidence or [])))
        st.markdown("**Verification**")
        st.caption("Not available — run failed.")
        st.markdown("**Runtime**")
        st.write(
            {
                "backend": result.backend,
                "model": result.model,
                "device": result.device,
                "error": result.error,
            }
        )
        st.markdown("**Retrieved evidence / scores / metadata**")
        render_evidence(result.retrieved_evidence or [])
        return

    overlay = uq_ui_confidence_overlay(result)
    decision = result.decision or "n/a"
    heading = overlay["decision_heading"] if overlay["show"] and overlay["decision_heading"] else decision
    if decision == "ABSTAIN":
        st.error(f"Decision: {heading}")
    else:
        st.success(f"Decision: {decision}")
    if overlay["show"] and overlay["warning"]:
        st.warning(overlay["warning"])
    if overlay["show"] and overlay["note"]:
        st.caption(overlay["note"])

    st.markdown("**Generated answer**")
    st.write(result.answer)

    if result.architecture == ARCHITECTURE_MULTI_AGENT_UQ and resolve_displayed_confidence(result) is None:
        st.error("UQ confidence could not be calculated. Displaying n/a — this is not a valid 0.0 confidence.")

    col_a, col_b, col_c, col_d = st.columns(4)
    # Do not use st.metric for 0–1 scores: Streamlit can render 0.7688 / 0.55 as 0.
    col_a.markdown("**Confidence**")
    col_a.write(format_confidence_display(result))
    col_b.markdown("**Threshold**")
    col_b.write(format_threshold_display(result))
    col_c.metric("Latency (s)", format_optional(result.latency_seconds))
    col_d.metric("Evidence chunks", str(len(result.retrieved_evidence or [])))

    st.markdown("**Verification**")
    verify = result.verification_result
    if not verify:
        st.caption("Not applicable for this architecture.")
    else:
        st.write(
            {
                "status": verify.get("status"),
                "rationale": verify.get("rationale"),
                "verification_score": verify.get("verification_score"),
                "lexical_score": verify.get("lexical_score"),
                "llm_score": verify.get("llm_score"),
                "verification_threshold": verify.get("verification_threshold"),
            }
        )
        if verify.get("rationale") and not str(verify.get("rationale", "")).startswith(str(verify.get("status") or "")):
            st.error("Verification status and rationale are inconsistent.")

    uq = (result.configuration or {}).get("uncertainty_result")
    if uq:
        st.markdown("**Uncertainty**")
        st.write(
            {
                "method": uq.get("method"),
                "retrieval_score": uq.get("retrieval_score"),
                "verification_score": uq.get("verification_score"),
                "confidence": uq.get("confidence"),
                "displayed_confidence": format_confidence_display(result),
                "displayed_threshold": format_threshold_display(result),
                "decision": result.decision,
            }
        )

    st.markdown("**Runtime**")
    st.write(
        {
            "backend": result.backend,
            "model": result.model,
            "quantisation": result.quantisation,
            "device": result.device,
            "gpu": result.gpu,
            "latency_seconds": result.latency_seconds,
        }
    )

    st.markdown("**Retrieved evidence / scores / metadata**")
    if result.retrieval_scores:
        st.caption("Retrieval scores: " + ", ".join(f"{s:.4f}" for s in result.retrieval_scores))
    render_evidence(result.retrieved_evidence or [])


### Viva explanation

Confidence is written with `st.write` because `st.metric` rendered 0.7688 as 0.

### Likely viva question

Where is verification and confidence displayed?

### Answer

`render_architecture()` in `streamlit_app.py`.

## PHASE 12 — Pilot (18 cases)

Notebook: `V2/notebooks/colab_phase12_pilot.ipynb`.

### MASTER CODE ID: M040

**Source file:** `V2/src/run/subset.py`
**Function/Class:** `PILOT_N_QUESTIONS`

### What this cell does

Pilot is first 6 frozen-140 rows × 3 architectures = 18 cases. Threshold note: smoke/demo NOT LOCKED.

### Libraries used

csv loader above.

### Inputs

Frozen 140 CSV.

### Outputs

6 question rows for `run_pilot`.

### Why this matters

Pilot is not 420. T was still 0.55.

In [ ]:
# Copied from V2/src/run/subset.py
"""Reproducible Phase 12 pilot subset from the frozen 140-question test set."""
from __future__ import annotations
import csv
import hashlib
import json
from pathlib import Path
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root

PILOT_N_QUESTIONS = 6
PILOT_N_ARCHITECTURES = 3
PILOT_N_CASES = PILOT_N_QUESTIONS * PILOT_N_ARCHITECTURES
SELECTION_RULE = "first_n_rows_of_frozen_140_csv"
THRESHOLD_NOTE = "smoke/demo — NOT LOCKED"


### Viva explanation

Phase 12 proved resume/JSONL on 18 cases before calibration.

### Likely viva question

How many pilot cases?

### Answer

18. Constants in `subset.py`: 6 questions × 3 architectures.

### MASTER CODE ID: M041

**Source file:** `V2/notebooks/colab_phase12_pilot.ipynb`
**Source notebook:** `V2/notebooks/colab_phase12_pilot.ipynb`
**Original notebook cell:** Cell 12
**Function/Class:** `Phase 12 pilot command`

### What this cell does

Original Colab cell that exported env and ran `scripts/run_pilot.py --backend llama_cpp`.

### Libraries used

llama_cpp via script.

### Inputs

Restored KB; smoke T=0.55.

### Outputs

18-case JSONL under `results/raw/phase12_pilot/`.

### Why this matters

Historical pilot. Do not run in viva.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 12 run_pilot
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 12 run_pilot')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print("import os\nos.environ['V2_REQUIRE_CUDA'] = '1'\nos.environ['V2_FORBID_MOCK'] = '1'\n!PYTHONPATH=. python scripts/run_pilot.py --backend llama_cpp --n-questions 6")
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 12 run_pilot)'
    )


### Viva explanation

Pilot used NOT LOCKED 0.55. Official lock is later.

### Likely viva question

Is the pilot the official evaluation?

### Answer

No. Official evaluation is Phase 15, 420 cases, T=0.65.

## PHASE 13 — DEV calibration / threshold lock

Notebook: `V2/notebooks/colab_phase13_calibration.ipynb`.

### MASTER CODE ID: M042

**Source file:** `V2/src/calibration/data.py`
**Function/Class:** `CALIBRATION_N / COVERAGE_FLOOR / SELECTION_RULE`

### What this cell does

Pre-registered rule constants: n=40, coverage floor 0.50, max selective accuracy, lowest-T tie-break.

### Libraries used

none beyond module constants.

### Inputs

Used by `select_threshold`.

### Outputs

Rule strings stored in the lock file.

### Why this matters

The T rule was declared before looking at TEST.

In [ ]:
# Copied from V2/src/calibration/data.py
"""Phase 13: load the frozen FinQA DEV calibration set (never the frozen 140)."""
from __future__ import annotations
import csv
import json
from pathlib import Path
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.data.select_calibration import rows_fingerprint
from src.run.subset import load_frozen_question_rows

CALIBRATION_N = 40
COVERAGE_FLOOR = 0.50
SELECTION_RULE = "max_selective_accuracy_coverage_ge_0.50"
TIE_BREAK = "lowest_threshold"


### Viva explanation

Coverage floor 0.50 means I would not pick a T that answers fewer than half of DEV.

### Likely viva question

What was the threshold selection rule?

### Answer

`max_selective_accuracy_coverage_ge_0.50` with tie_break `lowest_threshold`, DEV 40 only.

### MASTER CODE ID: M043

**Source file:** `V2/src/calibration/select.py`
**Function/Class:** `metrics_at_threshold / select_threshold`

### What this cell does

Sweeps T; among coverage ≥ 0.50 maximises selective accuracy; tie → lowest T. Uses draft text vs `program_answer`.

### Libraries used

`numeric_match`.

### Inputs

DEV UQ cases with confidence + draft.

### Outputs

selected threshold and curve. Official Colab lock: **0.65**.

### Why this matters

This is the calibration implementation. YAML `confidence_threshold` stays null; lock file is authoritative.

In [ ]:
# Copied from V2/src/calibration/select.py
"""Pre-registered DEV-only threshold selection. Never inspect the frozen 140."""
from __future__ import annotations
from typing import Any
from src.calibration.data import COVERAGE_FLOOR, SELECTION_RULE, TIE_BREAK
from src.evaluation.numeric import numeric_match

def draft_text(case: dict[str, Any]) -> str:
    cfg = case.get("configuration") or {}
    draft = cfg.get("draft_answer")
    if draft:
        return str(draft)
    return str(case.get("answer") or "")
def case_to_point(case: dict[str, Any]) -> dict[str, Any]:
    confidence = case.get("confidence")
    gold = case.get("reference_answer")
    predicted = draft_text(case)
    correct = numeric_match(predicted, gold) if confidence is not None else False
    return {
        "question_id": case.get("question_id"),
        "confidence": None if confidence is None else float(confidence),
        "correct": bool(correct),
        "gold": gold,
        "draft": predicted,
        "smoke_decision": case.get("decision"),
    }
def metrics_at_threshold(points: list[dict[str, Any]], threshold: float) -> dict[str, Any]:
    usable = [p for p in points if p.get("confidence") is not None]
    n = len(usable)
    answered = [p for p in usable if float(p["confidence"]) >= threshold]
    abstained = n - len(answered)
    n_answer = len(answered)
    n_correct = sum(1 for p in answered if p["correct"])
    coverage = (n_answer / n) if n else 0.0
    selective_accuracy = (n_correct / n_answer) if n_answer else 0.0
    return {
        "threshold": float(threshold),
        "n": n,
        "n_answer": n_answer,
        "n_abstain": abstained,
        "n_correct_answered": n_correct,
        "coverage": coverage,
        "selective_accuracy": selective_accuracy,
        "meets_coverage_floor": coverage >= COVERAGE_FLOOR,
    }
def sweep_thresholds(points: list[dict[str, Any]]) -> list[dict[str, Any]]:
    confidences = [float(p["confidence"]) for p in points if p.get("confidence") is not None]
    if not confidences:
        raise ValueError("No calibration confidences to sweep")
    return [metrics_at_threshold(points, t) for t in _candidate_thresholds(confidences)]
def select_threshold(points: list[dict[str, Any]]) -> dict[str, Any]:
    """Maximise selective accuracy among T with coverage >= 0.50; tie → lowest T."""
    curve = sweep_thresholds(points)
    feasible = [row for row in curve if row["meets_coverage_floor"]]
    if not feasible:
        return {
            "selected": False,
            "threshold": None,
            "rule": SELECTION_RULE,
            "coverage_floor": COVERAGE_FLOOR,
            "tie_break": TIE_BREAK,
            "reason": f"No threshold achieved coverage >= {COVERAGE_FLOOR}",
            "curve": curve,
        }
    best = max(feasible, key=lambda row: (row["selective_accuracy"], -row["threshold"]))
    return {
        "selected": True,
        "threshold": best["threshold"],
        "rule": SELECTION_RULE,
        "coverage_floor": COVERAGE_FLOOR,
        "tie_break": TIE_BREAK,
        "coverage": best["coverage"],
        "selective_accuracy": best["selective_accuracy"],
        "n_answer": best["n_answer"],
        "n_abstain": best["n_abstain"],
        "n": best["n"],
        "curve": curve,
    }


### Viva explanation

T=0.65 came from this DEV sweep. I did not pick it on the frozen 140. The 0.66 Streamlit band is UI-only, not a second research threshold.

### Likely viva question

Where is T selected and what is the locked value?

### Answer

`select_threshold()`; locked value `0.65` in `results/config/threshold.lock.json`.

### MASTER CODE ID: M044

**Source file:** `V2/src/calibration/lock.py`
**Function/Class:** `load_official_lock`

### What this cell does

Read-only load of the lock. Refuses if not locked, if TEST was used, if split≠dev, or if T≠0.65.

### Libraries used

json.

### Inputs

`results/config/threshold.lock.json`.

### Outputs

Payload including `threshold: 0.65`, `used_frozen_test_140: false`.

### Why this matters

Phase 14/15/20 cannot silently recalibrate.

In [ ]:
# Copied from V2/src/calibration/lock.py
"""Write or refuse ``threshold.lock.json``. Official lock requires a real DEV run."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Any
from src.calibration.data import CALIBRATION_N, assert_no_test_leakage
from src.calibration.select import case_to_point, select_threshold
from src.config import get_path, load_experiment_config, project_root
from src.run.store import utc_now
from src.run.subset import ids_sha256

def load_official_lock(path: Path | None = None) -> dict[str, Any]:
    """Read-only load of the Phase 13 DEV lock. Never writes or recalibrates."""
    dest = Path(path) if path is not None else lock_path()
    if not dest.is_file():
        raise FileNotFoundError(
            f"Official threshold lock missing at {dest}. Run Phase 13 Colab lock first."
        )
    payload = json.loads(dest.read_text(encoding="utf-8"))
    if payload.get("locked") is not True:
        raise RuntimeError("threshold.lock.json is not locked. Phase 14 will not invent a threshold.")
    if payload.get("used_frozen_test_140") is True:
        raise RuntimeError("Lock claims the frozen 140 was used. Refusing to run Phase 14.")
    if str(payload.get("source_split") or "") != "dev":
        raise RuntimeError("Official lock must have source_split=dev.")
    if int(payload.get("phase") or 0) < 13:
        raise RuntimeError("Official lock phase must be >= 13.")
    threshold = float(payload.get("threshold"))
    if abs(threshold - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError(
            f"Phase 14 requires locked T={EXPECTED_LOCKED_THRESHOLD}, found {threshold}. "
            "Do not recalibrate or modify the threshold."
        )
    payload["threshold"] = threshold
    return payload
def locked_threshold(path: Path | None = None) -> float:
    return float(load_official_lock(path)["threshold"])


### Viva explanation

The live artefact and the 420-case runner both call this. They will crash rather than invent a T.

### Likely viva question

How do you know T was not retuned on TEST?

### Answer

Lock has `used_frozen_test_140: false` and `source_split: dev`; `load_official_lock` enforces both.

### MASTER CODE ID: M045

**Source file:** `V2/notebooks/colab_phase13_calibration.ipynb`
**Source notebook:** `V2/notebooks/colab_phase13_calibration.ipynb`
**Original notebook cell:** Cell 11
**Function/Class:** `Phase 13 calibration command`

### What this cell does

Original Colab cell running DEV UQ calibration with llama_cpp (40 cases) to write the lock.

### Libraries used

llama_cpp via `run_calibration.py`.

### Inputs

Frozen DEV 40 + Phase 6 KB.

### Outputs

`threshold.lock.json`.

### Why this matters

Do not recalibrate in the viva.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 13 run_calibration
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 13 run_calibration')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print("import os\nos.environ['V2_REQUIRE_CUDA'] = '1'\nos.environ['V2_FORBID_MOCK'] = '1'\n!PYTHONPATH=. python scripts/run_calibration.py --backend llama_cpp --n-questions 40")
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 13 run_calibration)'
    )


### Viva explanation

I will point at the lock file, not rerun DEV generation.

### Likely viva question

Will you recalibrate T in the viva?

### Answer

No. Lock is already written. This cell is skipped.

In [ ]:
# Safe read of the existing lock (does not write or sweep T).
import json
from pathlib import Path
lock_path = Path("results/config/threshold.lock.json")
lock = json.loads(lock_path.read_text())
print({k: lock.get(k) for k in ["phase", "locked", "threshold", "rule", "coverage_floor", "coverage", "selective_accuracy", "n", "source_split", "used_frozen_test_140"]})


## PHASE 14 — Benchmark runner / 9-case validation

Notebook: `V2/notebooks/colab_phase14_benchmark_validation.ipynb`.

### MASTER CODE ID: M046

**Source file:** `V2/src/run/benchmark.py`
**Function/Class:** `select_benchmark_questions / verify_validation_subset / verify_full_subset`

### What this cell does

Phase 14 caps n=3 unless `allow_full`. Phase 15 requires n=140 and manifest SHA match. Mock refused when full.

### Libraries used

frozen CSV + sampling_manifest.json.

### Inputs

n_questions, allow_full flag.

### Outputs

Question rows. 9-case IDs are the first three frozen rows.

### Why this matters

Stops accidentally launching 420 from the validation notebook.

In [ ]:
# Copied from V2/src/run/benchmark.py
"""Phase 14/15 benchmark runner: frozen 140 × 3 architectures."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Any, Callable
from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock, lock_path
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.run.drive_sync import sync_benchmark_run
from src.run.store import CaseStore, case_is_successful, utc_now
from src.run.subset import ids_sha256, load_frozen_question_rows
from src.utils import create_run_id, get_logger

BENCHMARK_N_QUESTIONS = 140
BENCHMARK_N_CASES = BENCHMARK_N_QUESTIONS * BENCHMARK_N_ARCHITECTURES
VALIDATION_N_QUESTIONS = 3
VALIDATION_QUESTION_IDS = (
    "finqa_test_1000",
    "finqa_test_1012",
    "finqa_test_1017",
)
def select_benchmark_questions(
    *,
    n: int = VALIDATION_N_QUESTIONS,
    allow_full: bool = False,
    config: ExperimentConfig | None = None,
    csv_path: Path | None = None,
) -> list[dict[str, str]]:
    if n < 1:
        raise ValueError("Benchmark n must be >= 1")
    if allow_full:
        if n != BENCHMARK_N_QUESTIONS:
            raise ValueError(
                f"Full Phase 15 benchmark must use all {BENCHMARK_N_QUESTIONS} frozen questions "
                f"({BENCHMARK_N_CASES} cases)."
            )
    elif n > VALIDATION_N_QUESTIONS:
        raise ValueError(
            f"Phase 14 validation is capped at {VALIDATION_N_QUESTIONS} questions "
            f"({VALIDATION_N_CASES} cases). Do not launch the 420-case benchmark without "
            "--allow-full-420."
        )
    rows = load_frozen_question_rows(csv_path)
    if len(rows) != BENCHMARK_N_QUESTIONS:
        raise ValueError(f"Frozen test CSV has {len(rows)} rows; expected {BENCHMARK_N_QUESTIONS}")
    selected = rows[:n]
    frozen_ids = {row["id"] for row in rows}
    leaked_dev = [row["id"] for row in selected if str(row["id"]).startswith("finqa_dev_")]
    if leaked_dev:
        raise RuntimeError(f"Benchmark set includes DEV IDs: {leaked_dev[:5]}")
    if any(row["id"] not in frozen_ids for row in selected):
        raise ValueError("Benchmark subset is not a subset of the frozen 140")
    if any(not str(row["id"]).startswith("finqa_test_") for row in selected):
        raise RuntimeError("Benchmark IDs must be FinQA test (finqa_test_*)")
    return selected
def verify_validation_subset(questions: list[dict[str, str]]) -> None:
    ids = [row["id"] for row in questions]
    if ids != list(VALIDATION_QUESTION_IDS):
        raise ValueError(
            "Phase 14 9-case validation must be the first 3 frozen-140 rows "
            f"{list(VALIDATION_QUESTION_IDS)}; got {ids}"
        )
def verify_full_subset(questions: list[dict[str, str]]) -> None:
    from src.data.select_140 import rows_fingerprint

    ids = [row["id"] for row in questions]
    if len(ids) != BENCHMARK_N_QUESTIONS:
        raise ValueError(f"Full benchmark requires {BENCHMARK_N_QUESTIONS} questions, got {len(ids)}")
    manifest_path = project_root() / "data" / "final" / "sampling_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = list(manifest.get("selected_ids") or [])
    if ids != expected:
        raise ValueError("Full benchmark IDs do not match the frozen Phase 4 sampling manifest")
    expected_sha = str(manifest.get("selected_ids_sha256") or "")
    if expected_sha and rows_fingerprint([{"id": qid} for qid in ids]) != expected_sha:
        raise ValueError("Full benchmark ID SHA-256 does not match the Phase 4 manifest")


### Viva explanation

Phase 14 is engineering evidence only. I do not quote 9-case accuracy as the result.

### Likely viva question

Why is there a 9-case run and a 420-case run?

### Answer

9-case validated the runner+lock. Official evaluation is `allow_full=True`, 140×3.

### MASTER CODE ID: M047

**Source file:** `V2/src/run/benchmark.py`
**Function/Class:** `run_one_case / _runner_for`

### What this cell does

Dispatches each case to `run_single_agent` / `run_multi_agent` / `run_multi_agent_uq` independently; UQ gets locked T.

### Libraries used

the three RAG runners.

### Inputs

one frozen question dict + architecture name + locked threshold.

### Outputs

annotated `RAGCaseResult`.

### Why this matters

Shows architectures are not chained.

In [ ]:
# Copied from V2/src/run/benchmark.py
"""Phase 14/15 benchmark runner: frozen 140 × 3 architectures."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Any, Callable
from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock, lock_path
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.run.drive_sync import sync_benchmark_run
from src.run.store import CaseStore, case_is_successful, utc_now
from src.run.subset import ids_sha256, load_frozen_question_rows
from src.utils import create_run_id, get_logger

def _runner_for(architecture: str) -> Callable[..., RAGCaseResult]:
    runners = {
        ARCHITECTURE_SINGLE_AGENT: run_single_agent,
        ARCHITECTURE_MULTI_AGENT: run_multi_agent,
        ARCHITECTURE_MULTI_AGENT_UQ: run_multi_agent_uq,
    }
    return runners[architecture]
def run_one_case(
    question: dict[str, str],
    architecture: str,
    *,
    config: ExperimentConfig,
    backend: LLMBackend,
    backend_name: str,
    run_id: str,
    fingerprint: dict[str, Any],
    threshold: float,
    mode: str,
    phase: int = 14,
) -> RAGCaseResult:
    runner = _runner_for(architecture)
    kwargs: dict[str, Any] = {
        "question_id": question["id"],
        "reference_answer": question.get("program_answer") or None,
        "config": config,
        "backend": backend,
        "backend_name": backend_name,
        "run_id": run_id,
        "fingerprint": fingerprint,
    }
    if architecture == ARCHITECTURE_MULTI_AGENT_UQ:
        kwargs["threshold"] = threshold
    result = runner(question["question"], **kwargs)
    return annotate_benchmark_result(result, threshold=threshold, mode=mode, phase=phase)


### Viva explanation

The same question is sent separately to each runner. Multi-Agent does not see Single-Agent’s answer.

### Likely viva question

Do the three architectures share generated drafts?

### Answer

No. `_runner_for` calls three separate functions on the original question.

### MASTER CODE ID: M048

**Source file:** `V2/src/run/benchmark.py`
**Function/Class:** `run_benchmark (lock + dispatch)`

### What this cell does

Loads official lock, refuses mock on full 420, writes checkpointed JSONL, independent architectures.

### Libraries used

CaseStore, create_backend, load_official_lock.

### Inputs

backend_name, n_questions, allow_full.

### Outputs

raw `cases.jsonl` under phase14 or phase15 job folder.

### Why this matters

This is the benchmark execution engine. Do not call it here.

In [ ]:
# Copied from V2/src/run/benchmark.py
"""Phase 14/15 benchmark runner: frozen 140 × 3 architectures."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Any, Callable
from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock, lock_path
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.run.drive_sync import sync_benchmark_run
from src.run.store import CaseStore, case_is_successful, utc_now
from src.run.subset import ids_sha256, load_frozen_question_rows
from src.utils import create_run_id, get_logger

def run_benchmark(
    *,
    backend_name: str = "mock",
    n_questions: int = VALIDATION_N_QUESTIONS,
    allow_full: bool = False,
    config: ExperimentConfig | None = None,
    backend: LLMBackend | None = None,
    run_id: str | None = None,
    resume: str | None = None,
    resume_latest: bool = False,
    retry_failed: bool = True,
    stop_after: int | None = None,
    skip_preflight: bool = False,
    fingerprint: dict[str, Any] | None = None,
    lock_file: Path | None = None,
    sync_drive: bool = True,
) -> dict[str, Any]:
    """Run or resume the benchmark. Architectures stay independent (no chaining).

    Phase 14 validation: n_questions=3. Phase 15 official run: allow_full=True, n=140.
    """
    cfg = config or load_experiment_config()
    if allow_full and str(backend_name).lower() in {"mock", "test"}:
        raise RuntimeError("Full 420-case benchmark cannot use a mock backend.")
    lock = load_official_lock(lock_file)
    threshold = float(lock["threshold"])
    if abs(threshold - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError("Locked threshold mismatch. Do not recalibrate.")

    questions = select_benchmark_questions(n=n_questions, allow_full=allow_full, config=cfg)
    if n_questions == VALIDATION_N_QUESTIONS:
        verify_validation_subset(questions)
    elif n_questions == BENCHMARK_N_QUESTIONS:
        verify_full_subset(questions)

    phase_num = 15 if allow_full else 14
    job = "phase15_benchmark" if allow_full else "phase14_benchmark"
    run_prefix = "phase15" if allow_full else "phase14"
    mode = "benchmark" if allow_full else "benchmark_validation"
    question_ids = [row["id"] for row in questions]
    planned = planned_case_keys(question_ids)
    rid, run_dir, is_resume = resolve_run_dir(
        config=cfg,
        run_id=run_id,
        resume=resume,
        resume_latest=resume_latest,
        job=job,
        run_prefix=run_prefix,
    )
    checkpoint_copy = get_path(cfg, "results_checkpoints") / job / f"{rid}.json"
    store = CaseStore(run_dir, checkpoint_copy=checkpoint_copy)
    log = get_logger(run_id=rid, phase=run_prefix, architecture="benchmark")

    model_cfg = dict(cfg.section("model"))
    model_cfg["backend"] = backend_name
    fp = fingerprint or collect_fingerprint(model_config=model_cfg, project_root=str(project_root()))
    seed = cfg.section("execution").get("random_seed")
    llm = backend or create_backend(model_cfg)

    meta = {
        "phase": phase_num,
        "mode": mode,
        "run_id": rid,
        "backend": backend_name,
        "n_questions": len(questions),
        "n_architectures": BENCHMARK_N_ARCHITECTURES,
        "n_cases": len(planned),
        "question_ids": question_ids,
        "question_ids_sha256": ids_sha256(question_ids),
        "architectures": list(BENCHMARK_ARCHITECTURES),
        "independent_architectures": True,
        "chained": False,
        "threshold": threshold,
        "threshold_locked": True,
        "threshold_note": THRESHOLD_NOTE,
        "threshold_source": str(lock_file or lock_path()),
        "lock_run_id": lock.get("run_id"),
        "lock_source_split": lock.get("source_split"),
        "used_frozen_test_140_for_lock": False,
        "used_frozen_test_140_as_eval_set": True,
        "modifies_frozen_140": False,
        "modifies_frozen_calibration": False,
        "modifies_threshold_lock": False,
        "allow_full": allow_full,
        "job": job,
        "resumed": is_resume,
        "skip_preflight": skip_preflight,
        "device": fp.get("device"),
        "gpu": fp.get("gpu"),
        "random_seed": seed,
        "selection_rule": SELECTION_RULE,
    }
    store.write_checkpoint(meta, planned)
    log.info(
        "Benchmark start run_id=%s mode=%s resume=%s n_cases=%s completed=%s failed=%s pending=%s T=%.2f LOCKED",
        rid,
        mode,
        is_resume,
        len(planned),
        len(store.completed_keys),
        len(store.failed_keys),
        store.progress(planned)["n_pending"],
        threshold,
    )

    executed = 0
    skipped = 0
    questions_by_id = {row["id"]: row for row in questions}
    lock_before = Path(lock_file or lock_path()).read_text(encoding="utf-8") if Path(lock_file or lock_path()).is_file() else ""
    for case_key in planned:
        architecture, qid = case_key.split(":", 1)
        question = questions_by_id[qid]
        if not store.should_run(case_key, retry_failed=retry_failed):
            skipped += 1
            log.info("SKIP duplicate/completed case_key=%s", case_key)
            continue
        if stop_after is not None and executed >= stop_after:
            log.info("STOP_AFTER=%s reached; checkpoint saved for resume", stop_after)
            break
        log.info("RUN case_key=%s", case_key)
        try:
            result = run_one_case(
                question,
                architecture,
                config=cfg,
                backend=llm,
                backend_name=backend_name,
                run_id=rid,
                fingerprint=fp,
                threshold=threshold,
                mode=mode,
                phase=phase_num,
            )
        except Exception as exc:  # noqa: BLE001
            result = error_result(
                run_id=rid,
                question=question,
                architecture=architecture,
                error=f"{type(exc).__name__}: {exc}",
                fingerprint=fp,
                backend_name=backend_name,
                threshold=threshold,
                seed=seed,
                mode=mode,
                phase=phase_num,
            )
            log.info("ERROR case_key=%s error=%s", case_key, result.error)
        written = store.append_result(result)
        if not written:
            skipped += 1
            log.info("SKIP duplicate write case_key=%s", case_key)
        else:
            executed += 1
            ok = case_is_successful(result)
            log.info(
                "progress completed=%s failed=%s pending=%s | %s case_key=%s decision=%s n_evidence=%s confidence=%s threshold=%s latency=%.2fs error=%s",
                store.progress(planned)["n_completed"],
                store.progress(planned)["n_failed"],
                store.progress(planned)["n_pending"],
                "OK" if ok else "FAIL",
                case_key,
                result.decision,
                len(result.retrieved_evidence),
                result.confidence,
                result.threshold,
                result.latency_seconds,
                result.error,
            )
        store.write_checkpoint(meta, planned)
        if sync_drive:
            drive_info = sync_benchmark_run(
                run_dir, run_id=rid, checkpoint_copy=checkpoint_copy, job=job
            )
            if drive_info.get("synced"):
                log.info("Drive sync dest=%s", drive_info.get("dest"))

    lock_after_path = Path(lock_file or lock_path())
    if lock_before and lock_after_path.is_file() and lock_after_path.read_text(encoding="utf-8") != lock_before:
        raise RuntimeError("threshold.lock.json changed during the benchmark. Refusing to continue.")

    progress = store.progress(planned)
    status = "PASS" if progress["n_completed"] == len(planned) and progress["n_failed"] == 0 else (
        "INCOMPLETE" if progress["n_pending"] else "FAIL"
    )
    summary = {
        **meta,
        **progress,
        "status": status,
        "executed_this_session": executed,
        "skipped_this_session": skipped,
        "raw_path": _relative_or_absolute(store.raw_path),
        "checkpoint_path": str(store.checkpoint_path),
        "recorded_at_utc": utc_now(),
        "fingerprint": {
            "device": fp.get("device"),
            "gpu": fp.get("gpu"),
            "model_config": fp.get("model_config"),
            "git_commit": fp.get("git_commit"),
        },
    }
    (run_dir / "summary.json").write_text(
        json.dumps(summary, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    if sync_drive:
        summary["drive_sync"] = sync_benchmark_run(
            run_dir, run_id=rid, checkpoint_copy=checkpoint_copy, job=job
        )
    log.info(
        "Benchmark end status=%s completed=%s failed=%s pending=%s executed=%s skipped=%s T=%.2f LOCKED",
        status,
        progress["n_completed"],
        progress["n_failed"],
        progress["n_pending"],
        executed,
        skipped,
        threshold,
    )
    return summary


### Viva explanation

Phase 15 is this function with `allow_full=True`, n=140, backend llama_cpp.

### Likely viva question

Where is the 420-case loop?

### Answer

`run_benchmark()` in `V2/src/run/benchmark.py`, invoked by `scripts/run_full_benchmark.py`.

## PHASE 15 — Final 420-case benchmark

Notebook: `V2/notebooks/colab_phase15_full_benchmark.ipynb`.

### MASTER CODE ID: M049

**Source file:** `V2/scripts/run_full_benchmark.py`
**Function/Class:** `main`

### What this cell does

Phase 15 CLI: always 140×3, mock refused, uses locked T, separate raw folder so resume cannot pick Phase 14.

### Libraries used

argparse, `run_benchmark(..., allow_full=True)`.

### Inputs

`--backend llama_cpp` on Colab T4.

### Outputs

`results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/`.

### Why this matters

Official architecture benchmark. Not executed in this notebook.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 15 run_full_benchmark
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 15 run_full_benchmark')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('# Copied from V2/scripts/run_full_benchmark.py lines 33–55\ndef main() -> int:\n    parser = argparse.ArgumentParser(\n        description="Phase 15 final benchmark (140 questions × 3 architectures = 420 cases; T=0.65 locked)"\n    )\n    parser.add_argument("--backend", default="llama_cpp", help="llama_cpp|transformers (mock refused)")\n    parser.add_argument("--run-id", default=None)\n    parser.add_argument("--resume", default=None, help="Resume this run_id")\n    parser.add_argument("--resume-latest", action="store_true")\n    parser.add_argument("--no-retry-failed", action="store_true")\n    parser.add_argument("--stop-after", type=int, default=None, help="Execute at most N new cases then stop")\n    parser.add_argument("--skip-preflight", action="store_true")\n    parser.add_argument("--no-drive-sync", action="store_true")\n    args = parser.parse_args()\n\n    if str(args.backend).lower() in {"mock", "test"}:\n        raise SystemExit("Phase 15 refuses a mock backend. Use llama_cpp on Colab GPU.")\n\n    config = load_experiment_config()\n    setup_logging(\n        level="INFO",\n        log_dir=get_path(config, "results_logs"),\n        run_id=args.run_id or args.resume or "phase15",\n        console=True,\n')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 15 run_full_benchmark)'
    )


### Viva explanation

I will show the saved 420/420 summary, not rerun overnight generation.

### Likely viva question

Which script is the official 420-case job?

### Answer

`V2/scripts/run_full_benchmark.py` plus `colab_phase15_full_benchmark.ipynb`.

### MASTER CODE ID: M050

**Source file:** `V2/notebooks/colab_phase15_full_benchmark.ipynb`
**Source notebook:** `V2/notebooks/colab_phase15_full_benchmark.ipynb`
**Original notebook cell:** Cell 11
**Function/Class:** `Phase 15 notebook execution cell`

### What this cell does

Original Colab cell that launched the 420-case `run_full_benchmark.py` job.

### Libraries used

llama_cpp on T4.

### Inputs

Restored KB + lock T=0.65.

### Outputs

420 JSONL rows.

### Why this matters

Copied for walkthrough. Skipped under VIVA_MODE.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 15 420-case notebook cell
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 15 420-case notebook cell')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print("import os\nos.environ['V2_REQUIRE_CUDA'] = '1'\nos.environ['V2_FORBID_MOCK'] = '1'\nos.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'\n!PYTHONPATH=. python scripts/run_full_benchmark.py --backend llama_cpp")
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 15 420-case notebook cell)'
    )


### Viva explanation

That cell is how I ran the official job on Colab. Today we inspect artefacts.

### Likely viva question

Will this notebook rerun 420 cases?

### Answer

No. VIVA_MODE skips this cell.

## PHASE 16 — Evaluation + metrics (CPU) and LLM-as-judge

Notebook: `V2/notebooks/colab_phase16_judge.ipynb` for the judge pass. CPU scoring is `scripts/run_evaluation.py`.

### MASTER CODE ID: M051

**Source file:** `V2/src/evaluation/numeric.py`
**Function/Class:** `numeric_match / parse_numbers`

### What this cell does

Primary correctness: any parsed number in the prediction matches `program_answer` within 1% relative / 1e-4 absolute. `%` also tried as /100.

### Libraries used

math, re.

### Inputs

predicted text, gold `program_answer`.

### Outputs

bool.

### Why this matters

Displayed correctness and claim correctness both use this.

In [ ]:
# Copied from V2/src/evaluation/numeric.py
"""Numeric match against FinQA ``program_answer`` (calibration / later metrics)."""
from __future__ import annotations
import math
import re

def parse_numbers(text: str | None) -> list[float]:
    """Extract numeric tokens. Values with % are stored as given and as /100."""
    if not text:
        return []
    values: list[float] = []
    for match in _NUMBER_RE.finditer(str(text)):
        raw = match.group(0).strip()
        percent = raw.endswith("%")
        cleaned = raw.replace(",", "").replace("%", "").strip()
        try:
            number = float(cleaned)
        except ValueError:
            continue
        values.append(number)
        if percent:
            values.append(number / 100.0)
        else:
            values.append(number / 100.0)
            values.append(number * 100.0)
    return values
def numeric_match(
    predicted: str | None,
    gold: str | None,
    *,
    rel_tol: float = 0.01,
    abs_tol: float = 1e-4,
) -> bool:
    """True if any number in ``predicted`` matches ``gold`` within tolerance."""
    if gold is None or str(gold).strip() == "":
        return False
    gold_values = parse_numbers(str(gold))
    if not gold_values:
        return False
    gold_number = gold_values[0]
    for predicted_number in parse_numbers(predicted):
        if math.isclose(predicted_number, gold_number, rel_tol=rel_tol, abs_tol=abs_tol):
            return True
    return False


### Viva explanation

Correctness is numeric match to FinQA `program_answer`, not BLEU and not official RAGAS.

### Likely viva question

What is your accuracy metric?

### Answer

`numeric_match()` against `program_answer` in `V2/src/evaluation/numeric.py`.

### MASTER CODE ID: M052

**Source file:** `V2/src/evaluation/metrics.py`
**Function/Class:** `score_case / scored_claim_text`

### What this cell does

CPU scoring of a saved case: displayed vs claim correctness, coverage/answered, unsupported_emitted, context P/R. No GPU.

### Libraries used

numeric_match, token_overlap.

### Inputs

Phase 15 case dict + gold CSV row.

### Outputs

Metric row. UQ claim uses `draft_answer` when abstaining.

### Why this matters

Unsupported-emitted = answered AND displayed numeric match failed. Operational, not a hallucination oracle.

In [ ]:
# Copied from V2/src/evaluation/metrics.py
"""Phase 16 CPU metrics from saved RAG cases. No LLM / GPU / new generation."""
from __future__ import annotations
from typing import Any
from src.evaluation.numeric import numeric_match
from src.rag.text_utils import token_overlap

def scored_claim_text(case: dict[str, Any]) -> str:
    """Text used for correctness/faithfulness of the model's claim.

    UQ ABSTAIN replaces the displayed answer with the abstention template;
    the draft is the claim that would have been emitted.
    """
    cfg = case.get("configuration") or {}
    draft = cfg.get("draft_answer")
    if case.get("architecture") == ARCHITECTURE_UQ and draft:
        return str(draft)
    return str(case.get("answer") or "")
def displayed_text(case: dict[str, Any]) -> str:
    return str(case.get("answer") or "")
def score_case(case: dict[str, Any], gold: dict[str, str]) -> dict[str, Any]:
    """Score one saved architecture–question case. CPU only."""
    chunks = list(case.get("retrieved_evidence") or [])
    gold_program = gold.get("program_answer") or case.get("reference_answer")
    gold_original = gold.get("original_answer") or ""
    claim = scored_claim_text(case)
    displayed = displayed_text(case)
    evidence = evidence_text(chunks)
    decision = str(case.get("decision") or "ANSWER")
    answered = decision == "ANSWER"

    correct_claim = numeric_match(claim, gold_program)
    correct_displayed = numeric_match(displayed, gold_program)
    correct_original = numeric_match(claim, gold_original) if gold_original else False

    faithfulness = token_overlap(claim, evidence)
    stored_verify = case.get("verification_result") if isinstance(case.get("verification_result"), dict) else {}
    precision = context_precision(chunks, gold)
    recall = context_recall(chunks, gold)
    recall_numeric = numeric_match(evidence, gold_program)

    unsupported_emitted = bool(answered and not correct_displayed)

    return {
        "case_key": case.get("case_key") or f"{case.get('architecture')}:{case.get('question_id')}",
        "run_id": case.get("run_id"),
        "question_id": case.get("question_id"),
        "architecture": case.get("architecture"),
        "decision": decision,
        "answered": answered,
        "confidence": case.get("confidence"),
        "threshold": case.get("threshold"),
        "n_evidence": len(chunks),
        "gold_program_answer": gold_program,
        "gold_file_name": gold.get("file_name"),
        "gold_context_id": gold.get("context_id"),
        "answer_correctness": int(correct_displayed),
        "answer_correctness_claim": int(correct_claim),
        "answer_correctness_original_answer": int(correct_original),
        "faithfulness": faithfulness,
        "faithfulness_stored_verification_score": stored_verify.get("verification_score"),
        "faithfulness_stored_lexical_score": stored_verify.get("lexical_score"),
        "context_precision": precision,
        "context_recall": recall,
        "context_recall_numeric": int(recall_numeric),
        "unsupported_emitted": int(unsupported_emitted),
        "latency_seconds": case.get("latency_seconds"),
        "backend": case.get("backend"),
        "device": case.get("device"),
        "gpu": case.get("gpu"),
        "model": case.get("model"),
        "quantisation": case.get("quantisation"),
        "error": case.get("error"),
        "used_llm_inference": False,
        "used_gpu": False,
    }


### Viva explanation

When UQ abstains, displayed correctness can be 0 while claim correctness still scores the draft.

### Likely viva question

How is unsupported-emitted defined?

### Answer

`unsupported_emitted = answered and not correct_displayed` in `score_case()`.

### MASTER CODE ID: M053

**Source file:** `V2/src/evaluation/judge.py`
**Function/Class:** `METRIC_LABEL / claim_for_judge / prompt_contains_forbidden / judge_one_case`

### What this cell does

Post-hoc faithfulness judge on saved evidence+claim. Label is custom/RAGAS-inspired. No gold context/answer in the prompt. Does not rerun RAG.

### Libraries used

Qwen via llama_cpp; parse_unit_score.

### Inputs

Saved Phase 15 case; temperature 0, max_new_tokens 32.

### Outputs

parsed_faithfulness_score 0–1; `used_rag_rerun=False`.

### Why this matters

Separate 420-call job. Not the official RAGAS library.

In [ ]:
# Copied from V2/src/evaluation/judge.py
"""Post-hoc LLM-as-judge faithfulness over saved Phase 15 cases."""
from __future__ import annotations
import hashlib
from typing import Any
from src.models.types import LLMBackend
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ
from src.rag.text_utils import parse_unit_score

METRIC_LABEL = "LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)"
PROMPT_ID = "phase16_judge_faithfulness_v1"
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_NEW_TOKENS = 32
def claim_for_judge(case: dict[str, Any]) -> tuple[str, str]:
    """Return (claim_text, claim_source). UQ uses the draft, not the abstention template."""
    if case.get("architecture") == ARCHITECTURE_MULTI_AGENT_UQ:
        draft = (case.get("configuration") or {}).get("draft_answer")
        return str(draft or ""), "draft_answer"
    return str(case.get("answer") or ""), "answer"
def build_judge_prompt(*, question: str, evidence: str, claim: str) -> str:
    user = JUDGE_USER_TEMPLATE.format(
        evidence=evidence,
        question=(question or "").strip(),
        claim=(claim or "").strip(),
    )
    return f"{JUDGE_SYSTEM.strip()}\n\n{user.strip()}"
def prompt_contains_forbidden(prompt: str, case: dict[str, Any]) -> list[str]:
    """Detect accidental leakage of gold context or gold answers into the judge prompt."""
    hits: list[str] = []
    gold_context = str((case.get("gold_context") or "")).strip()
    if gold_context and len(gold_context) > 40 and gold_context in prompt:
        hits.append("gold_context")
    for key in ("program_answer", "original_answer", "reference_answer"):
        value = str(case.get(key) or "").strip()
        if value and len(value) >= 4 and value in prompt and value not in (case.get("answer") or ""):
            if value not in ((case.get("configuration") or {}).get("draft_answer") or ""):
                if value not in format_retrieved_evidence(list(case.get("retrieved_evidence") or [])):
                    hits.append(key)
    lowered = prompt.lower()
    if "program_answer" in lowered or "gold context" in lowered:
        hits.append("forbidden_label")
    return hits
def judge_one_case(
    case: dict[str, Any],
    llm: LLMBackend,
    *,
    source_raw_sha256: str,
    temperature: float = JUDGE_TEMPERATURE,
    max_new_tokens: int = JUDGE_MAX_NEW_TOKENS,
    n_ctx: int = JUDGE_N_CTX,
    fingerprint: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Score one saved case. Does not retrieve or regenerate the RAG answer."""
    fp = fingerprint or {}
    gpu = fp.get("gpu")
    gpu_name = gpu.get("name") if isinstance(gpu, dict) else gpu
    claim, claim_source = claim_for_judge(case)
    chunks = list(case.get("retrieved_evidence") or [])
    evidence = format_retrieved_evidence(chunks)
    question = str(case.get("question") or "")
    prompt = build_judge_prompt(question=question, evidence=evidence, claim=claim)
    leaked = prompt_contains_forbidden(prompt, case)
    record: dict[str, Any] = {
        "case_key": case.get("case_key") or f"{case.get('architecture')}:{case.get('question_id')}",
        "question_id": case.get("question_id"),
        "architecture": case.get("architecture"),
        "source_raw_sha256": source_raw_sha256,
        "claim_source": claim_source,
        "decision": case.get("decision"),
        "judge_model": "Qwen3-8B",
        "judge_metric_label": METRIC_LABEL,
        "backend": getattr(llm, "name", None),
        "device": fp.get("device"),
        "gpu": gpu_name,
        "quantisation": "Q4_K_M" if getattr(llm, "name", "") == "llama_cpp" else getattr(llm, "name", None),
        "n_ctx": n_ctx,
        "temperature": temperature,
        "max_new_tokens": max_new_tokens,
        "prompt_id": PROMPT_ID,
        "prompt_hash": prompt_hash(),
        "raw_judge_output": None,
        "parsed_faithfulness_score": None,
        "parse_failure": False,
        "latency_seconds": None,
        "used_rag_rerun": False,
        "used_gold_context": False,
        "used_gold_answer": False,
        "error": None,
    }
    if leaked:
        record["error"] = f"judge_prompt_leak:{','.join(leaked)}"
        record["parse_failure"] = True
        return record
    if not claim.strip():
        record["error"] = "empty_claim"
        record["parse_failure"] = True
        return record
    if not any(str(c.get("text") or c.get("content") or "").strip() for c in chunks):
        record["error"] = "empty_retrieved_evidence"
        record["parse_failure"] = True
        return record
    try:
        gen = llm.generate(prompt, temperature=temperature, max_new_tokens=max_new_tokens)
    except Exception as exc:  # noqa: BLE001
        record["error"] = str(exc)
        record["parse_failure"] = True
        return record
    raw = gen.text or ""
    record["raw_judge_output"] = raw
    record["latency_seconds"] = gen.latency_seconds
    record["backend"] = gen.backend or record["backend"]
    record["quantisation"] = gen.quantisation or record["quantisation"]
    parsed = parse_unit_score(raw)
    if parsed is None:
        record["parse_failure"] = True
        record["error"] = "parse_failure"
        return record
    record["parsed_faithfulness_score"] = float(parsed)
    record["parse_failure"] = False
    record["error"] = None
    return record


### Viva explanation

The judge sees retrieved chunks and the claim only. UQ claim is the draft, not the abstention sentence.

### Likely viva question

Did you use official RAGAS?

### Answer

No. `METRIC_LABEL` is ‘LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)’.

### MASTER CODE ID: M054

**Source file:** `V2/notebooks/colab_phase16_judge.ipynb`
**Source notebook:** `V2/notebooks/colab_phase16_judge.ipynb`
**Original notebook cell:** Cell 11
**Function/Class:** `Phase 16 judge notebook cell`

### What this cell does

Original Colab cell that launched the 420-case judge (`run_judge.py`).

### Libraries used

llama_cpp.

### Inputs

Frozen Phase 15 JSONL (not regenerated).

### Outputs

`phase16_judge_20260828T152623Z_06661255/judge.jsonl`.

### Why this matters

Do not rerun.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 16 run_judge
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 16 run_judge')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print("import os\nos.environ['V2_REQUIRE_CUDA'] = '1'\nos.environ['V2_FORBID_MOCK'] = '1'\nos.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'\n!PYTHONPATH=. python scripts/run_judge.py --backend llama_cpp")
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 16 run_judge)'
    )


### Viva explanation

Judge is post-hoc over saved cases.

### Likely viva question

Does the judge regenerate RAG answers?

### Answer

No. `used_rag_rerun` is false; it scores saved claims.

## PHASE 17 — Statistics on frozen Phase 15/16 results

### MASTER CODE ID: M055

**Source file:** `V2/src/statistics/tests.py`
**Function/Class:** `mcnemar_exact`

### What this cell does

Exact McNemar on paired binary outcomes (n=140 questions). Used for RQ1 displayed correctness SA vs MA.

### Libraries used

numpy, scipy.stats.

### Inputs

Two length-140 0/1 series.

### Outputs

p_value, discordant counts, odds ratio. Recorded RQ1 p=0.6776 (not significant).

### Why this matters

Statistical unit is the question, paired across architectures. No RAG rerun.

In [ ]:
# Copied from V2/src/statistics/tests.py
"""Assumption-aware tests for Phase 17. No RAG / LLM imports."""
from __future__ import annotations
import math
from typing import Any
import numpy as np
from scipy import stats
from src.statistics.constants import ALPHA, BOOTSTRAP_N, BOOTSTRAP_SEED

def mcnemar_exact(left: list[int], right: list[int]) -> dict[str, Any]:
    """Exact McNemar test on paired binary outcomes (same questions).

    Table uses left/right correctness:
    n11 both 1, n10 left 1 right 0, n01 left 0 right 1, n00 both 0.
    """
    a = np.asarray(left, dtype=int)
    b = np.asarray(right, dtype=int)
    if a.size != b.size:
        raise ValueError("McNemar requires equal-length paired series")
    n = int(a.size)
    n11 = int(np.sum((a == 1) & (b == 1)))
    n10 = int(np.sum((a == 1) & (b == 0)))
    n01 = int(np.sum((a == 0) & (b == 1)))
    n00 = int(np.sum((a == 0) & (b == 0)))
    discordant = n10 + n01
    if discordant == 0:
        exact_p = 1.0
        chi2 = 0.0
        chi2_p = 1.0
    else:
        exact_p = float(stats.binomtest(n01, n=discordant, p=0.5, alternative="two-sided").pvalue)
        chi2 = (abs(n01 - n10) - 1) ** 2 / discordant
        chi2_p = float(stats.chi2.sf(chi2, 1))
    odds_num = n01 + 0.5
    odds_den = n10 + 0.5
    odds_ratio = odds_num / odds_den
    cohen_g = (n01 / discordant - 0.5) if discordant else 0.0
    mean_left = n11 / n + n10 / n
    mean_right = n11 / n + n01 / n
    return {
        "n": n,
        "n11_both_positive": n11,
        "n10_left_only": n10,
        "n01_right_only": n01,
        "n00_both_negative": n00,
        "n_discordant": discordant,
        "mean_left": mean_left,
        "mean_right": mean_right,
        "mean_difference": mean_right - mean_left,
        "test": "McNemar exact (binomial, two-sided)",
        "statistic": n01,
        "statistic_name": "n01 (right-only positives among discordant pairs)",
        "df": None,
        "p_value": exact_p,
        "chi2_continuity": chi2,
        "chi2_p_value": chi2_p,
        "effect_odds_ratio_haldane": odds_ratio,
        "effect_cohens_g": cohen_g,
        "ci_left": wilson_ci(n11 + n10, n),
        "ci_right": wilson_ci(n11 + n01, n),
    }


### Viva explanation

McNemar is appropriate because the same 140 questions are compared. I do not claim Multi-Agent improved accuracy; p=0.6776.

### Likely viva question

Which test is RQ1?

### Answer

`mcnemar_exact` on displayed numeric correctness, SA vs MA, in `tests.py`, called from `analyse()`.

### MASTER CODE ID: M056

**Source file:** `V2/src/statistics/analysis.py`
**Function/Class:** `analyse (RQ1 confirmatory call)`

### What this cell does

Loads joined Phase 16+judge tables and runs confirmatory McNemar SA vs MA displayed correctness.

### Libraries used

statistics.tests, statistics.load.

### Inputs

Frozen processed cases + judge scores. CPU only.

### Outputs

Test rows written to `phase17_tests.csv`.

### Why this matters

Shows the exact RQ1 confirmatory test wiring.

In [ ]:
# DO NOT RUN DURING VIVA — DO NOT RUN DURING VIVA: Phase 17 analyse() on full joined tables
if VIVA_MODE:
    print('DO NOT RUN DURING VIVA: Phase 17 analyse() on full joined tables')
    print('Original cell kept for walkthrough. Not executed.')
    print('----- original -----')
    print('# Copied from V2/src/statistics/analysis.py lines 111–145\ndef analyse(root=None) -> dict[str, Any]:\n    joined = load_joined(root)\n    descriptive = _arch_descriptive(joined)\n\n    sa_disp = _i(series(joined, ARCH_SA, "answer_correctness"))\n    ma_disp = _i(series(joined, ARCH_MA, "answer_correctness"))\n    uq_disp = _i(series(joined, ARCH_UQ, "answer_correctness"))\n    sa_claim = _i(series(joined, ARCH_SA, "answer_correctness_claim"))\n    ma_claim = _i(series(joined, ARCH_MA, "answer_correctness_claim"))\n    uq_claim = _i(series(joined, ARCH_UQ, "answer_correctness_claim"))\n    sa_unsup = _i(series(joined, ARCH_SA, "unsupported_emitted"))\n    ma_unsup = _i(series(joined, ARCH_MA, "unsupported_emitted"))\n    uq_unsup = _i(series(joined, ARCH_UQ, "unsupported_emitted"))\n    sa_llm = _f(series(joined, ARCH_SA, "llm_faithfulness"))\n    ma_llm = _f(series(joined, ARCH_MA, "llm_faithfulness"))\n    uq_llm = _f(series(joined, ARCH_UQ, "llm_faithfulness"))\n    sa_ov = _f(series(joined, ARCH_SA, "faithfulness"))\n    ma_ov = _f(series(joined, ARCH_MA, "faithfulness"))\n    uq_ov = _f(series(joined, ARCH_UQ, "faithfulness"))\n    uq_ans = _i(series(joined, ARCH_UQ, "answered"))\n    uq_conf = _f(series(joined, ARCH_UQ, "confidence"))\n\n    # RQ1 confirmatory: SA vs MA displayed numeric correctness\n    rq1_primary = dict(mcnemar_exact(sa_disp, ma_disp))\n    rq1_primary.update({\n        "id": "rq1_mcnemar_displayed_sa_vs_ma",\n        "rq": "RQ1",\n        "role": "confirmatory",\n        "left": ARCH_SA,\n        "right": ARCH_MA,\n        "outcome": "displayed numeric answer correctness",\n        "layer": "numeric FinQA correctness (primary RQ1)",\n        "unit": "frozen FinQA test question (n=140), paired across architectures",\n    })\n    rq1_conf = _annotate_family([rq1_primary], "rq1_confirmatory")\n')
else:
    raise RuntimeError(
        'Refusing to execute this historical research cell from the viva notebook. '
        'Use the original file instead. (DO NOT RUN DURING VIVA: Phase 17 analyse() on full joined tables)'
    )


### Viva explanation

RQ2 uses Spearman of UQ confidence vs judge faithfulness; RQ3 uses McNemar on unsupported-emitted.

### Likely viva question

Where are the RQ tests assembled?

### Answer

`analyse()` in `V2/src/statistics/analysis.py`.

## PHASE 18 — Qualitative error analysis of the frozen 420-case benchmark

### MASTER CODE ID: M057

**Source file:** `V2/src/error_analysis/taxonomy.py`
**Function/Class:** `assign_category`

### What this cell does

Rule-based primary category: abstention vs retrieval_failure vs incorrect_numerical_reasoning vs unsupported_claim, etc. Numeric error is never labelled hallucination.

### Libraries used

parse_numbers.

### Inputs

A scored case row (answered, displayed_correct, context_recall, llm_faithfulness, …).

### Outputs

primary_category, error_layer, tags.

### Why this matters

Qualitative analysis on frozen outputs. Seed 18 sample in the pipeline, not shown here.

In [ ]:
# Copied from V2/src/error_analysis/taxonomy.py
"""Deterministic error taxonomy from recorded fields. No invented causes."""
from __future__ import annotations
from typing import Any
from src.evaluation.numeric import parse_numbers
from src.error_analysis.constants import FAITHFULNESS_LOW, HIGH_CONFIDENCE, PRIMARY_CATEGORIES

def assign_category(case: dict[str, Any]) -> dict[str, Any]:
    """Assign one primary category plus factual tags.

    Order is explicit so UQ abstention is not labelled as a displayed numeric error.
    Numeric incorrectness is never labelled hallucination.
    """
    answered = bool(case["answered"])
    displayed_ok = int(case["displayed_correct"]) == 1
    claim_ok = int(case["claim_correct"]) == 1
    recall = float(case["context_recall"])
    gold_in_evidence = int(case["context_recall_numeric"]) == 1
    llm = float(case["llm_faithfulness"])
    displayed = str(case.get("displayed_answer") or "")
    conf = case.get("confidence")
    verify = case.get("verification_status")

    tags: list[str] = []
    if recall == 0.0:
        tags.append("gold_file_or_context_absent_from_topk")
    if gold_in_evidence:
        tags.append("gold_number_present_in_evidence")
    else:
        tags.append("gold_number_absent_from_evidence")
    if float(case["context_precision"]) <= 0.25:
        tags.append("low_context_precision")
    if answered and conf is not None and float(conf) >= HIGH_CONFIDENCE and not displayed_ok:
        tags.append("false_confidence")
    if verify == "VERIFIED" and not claim_ok:
        tags.append("verification_false_positive")
    if verify == "VERIFIED" and claim_ok:
        tags.append("verification_true_positive")
    if verify == "WEAK_EVIDENCE" and not claim_ok:
        tags.append("verification_true_negative")
    if verify == "WEAK_EVIDENCE" and claim_ok:
        tags.append("verification_false_negative")
    if answered and not _has_number(displayed):
        tags.append("displayed_text_has_no_parsed_number")
    if llm < FAITHFULNESS_LOW:
        tags.append("low_llm_judge_faithfulness")
    else:
        tags.append("high_llm_judge_faithfulness")

    if not answered and claim_ok:
        primary = "incorrect_abstention"
    elif not answered and not claim_ok:
        primary = "appropriate_abstention"
    elif displayed_ok:
        primary = "correct_answer"
    elif recall == 0.0:
        primary = "retrieval_failure"
    elif not _has_number(displayed):
        primary = "non_numeric_answer"
    elif gold_in_evidence:
        primary = "incorrect_numerical_reasoning"
    elif llm < FAITHFULNESS_LOW:
        primary = "unsupported_claim"
    else:
        primary = "incorrect_despite_partial_evidence"

    if primary not in PRIMARY_CATEGORIES:
        raise ValueError(primary)

    layer = {
        "correct_answer": "numeric_correct",
        "appropriate_abstention": "abstention",
        "incorrect_abstention": "abstention",
        "retrieval_failure": "retrieval",
        "non_numeric_answer": "answer_format",
        "incorrect_numerical_reasoning": "numeric_error",
        "unsupported_claim": "unsupported_emission",
        "incorrect_despite_partial_evidence": "numeric_error",
    }[primary]

    return {
        "primary_category": primary,
        "error_layer": layer,
        "tags": tags,
    }


### Viva explanation

I do not call a wrong number a hallucination if the gold amount is in the evidence — that is incorrect numerical reasoning.

### Likely viva question

Where is error categorisation implemented?

### Answer

`assign_category()` in `V2/src/error_analysis/taxonomy.py`.

## PHASE 19 — Final reproducibility and research-integrity audit

### MASTER CODE ID: M058

**Source file:** `V2/src/audit/checks.py`
**Function/Class:** `run_audit (header / frozen pins)`

### What this cell does

Read-only audit importing SHA pins and lock loader. Explicitly avoids generation imports.

### Libraries used

json, hashlib via statistics.load.

### Inputs

Frozen CSV/lock/JSONL paths.

### Outputs

PASS/FAIL/NEEDS VERIFICATION rows.

### Why this matters

Integrity of 40 DEV → T=0.65 → 140 → 420 → Phase 16/17/18.

In [ ]:
# Copied from V2/src/audit/checks.py lines 1–37
"""Read-only consistency checks for the frozen research chain."""

from __future__ import annotations

import csv
import json
import subprocess
from collections import Counter
from pathlib import Path
from typing import Any

from src.audit import verify_audit_does_not_import_generation
from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock
from src.config import project_root
from src.statistics.constants import (
    ARCHITECTURES,
    CAL_40_REL,
    EXPECTED_CAL40_SHA256,
    EXPECTED_FROZEN140_SHA256,
    EXPECTED_JUDGE_SHA256,
    EXPECTED_LOCK_SHA256,
    EXPECTED_PHASE15_SHA256,
    EXPECTED_PROCESSED_SHA256,
    FROZEN_140_REL,
    JUDGE_METRIC_LABEL,
    JUDGE_REL,
    LOCKED_T,
    LOCK_REL,
    PHASE15_REL,
    PROCESSED_REL,
)
from src.statistics.load import sha256_file, verify_frozen_hashes
from src.run.subset import ids_sha256

PASS = "PASS"
FAIL = "FAIL"
NV = "NEEDS VERIFICATION"


### Viva explanation

The audit does not call Qwen. It checks files still match pinned SHA-256 values.

### Likely viva question

How do you show the freeze was not edited after results?

### Answer

Phase 19 audit + SHA pins in `src/statistics/constants.py`.

## PHASE 20 — Final live artefact

### MASTER CODE ID: M059

**Source file:** `V2/src/rag/live.py`
**Function/Class:** `resolve_live_locked_threshold`

### What this cell does

Live path loads T from the lock file; ignores yaml smoke 0.55; refuses TEST-tuned locks.

### Libraries used

calibration.lock.

### Inputs

`threshold.lock.json`.

### Outputs

float 0.65.

### Why this matters

Phase 20 artefact uses the research lock, not Phase 11 smoke T.

In [ ]:
# Copied from V2/src/rag/live.py
"""Live comparison runner: three independent RAG architectures on one question."""
from __future__ import annotations
import csv
import hashlib
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any
from src.config import ExperimentConfig, get_path, load_experiment_config, project_root
from src.models.factory import create_backend
from src.models.fingerprint import collect_fingerprint
from src.models.runtime_guard import LiveRuntimeError, mock_forbidden
from src.models.types import LLMBackend
from src.rag.multi_agent import run_multi_agent
from src.rag.multi_agent_uq import run_multi_agent_uq
from src.rag.schema import (
    ARCHITECTURE_MULTI_AGENT,
    ARCHITECTURE_MULTI_AGENT_UQ,
    ARCHITECTURE_SINGLE_AGENT,
    RAGCaseResult,
)
from src.rag.single_agent import run_single_agent
from src.utils import create_run_id

def resolve_live_locked_threshold() -> float:
    """Official T from threshold.lock.json. Does not retune or read yaml smoke_threshold."""
    from src.calibration.lock import EXPECTED_LOCKED_THRESHOLD, load_official_lock

    lock = load_official_lock()
    threshold = float(lock["threshold"])
    if abs(threshold - EXPECTED_LOCKED_THRESHOLD) > 1e-9:
        raise RuntimeError(
            f"Live artefact requires locked T={EXPECTED_LOCKED_THRESHOLD}, found {threshold}."
        )
    if lock.get("used_frozen_test_140") is True:
        raise RuntimeError("Lock claims the frozen 140 was used. Refusing live demo.")
    if str(lock.get("source_split") or "") != "dev":
        raise RuntimeError("Official lock must have source_split=dev.")
    return threshold
def annotate_live_uq_lock(result: RAGCaseResult, locked_t: float) -> RAGCaseResult:
    """Live-layer label only. Does not change retrieve/generate/verify internals."""
    result.threshold = float(locked_t)
    cfg = dict(result.configuration or {})
    cfg["threshold_source"] = "locked"
    cfg["threshold_locked"] = True
    cfg["threshold_note"] = "LOCKED T from results/config/threshold.lock.json (DEV 40 only)"
    result.configuration = cfg
    return result


### Viva explanation

Streamlit always uses locked 0.65. The 0.66 warning is display-only.

### Likely viva question

Which threshold does the live app use?

### Answer

`resolve_live_locked_threshold()` → 0.65 from the lock file.

### MASTER CODE ID: M060

**Source file:** `V2/app/streamlit_app.py`
**Function/Class:** `main`

### What this cell does

Three pages: Live RAG Demo, Benchmark Results, Benchmark Questions. `st.set_page_config` title `V2 RAG Artefact`.

### Libraries used

streamlit.

### Inputs

Sidebar radio `app_page`.

### Outputs

One of the three page renderers.

### Why this matters

This is the artefact navigation.

In [ ]:
# Copied from V2/app/streamlit_app.py
"""V2 live artefact: run all three RAG architectures on a fresh or frozen question."""
from __future__ import annotations
import json
import os
import sys
from pathlib import Path
import streamlit as st
from app.benchmark_ui import render_benchmark_questions_page, render_benchmark_results_page
from src.config import get_path, load_experiment_config, project_root
from src.rag.benchmark_catalogue import (
    apply_catalogue_prefill_to_live_input,
    apply_pending_app_page,
)
from src.models.factory import create_backend
from src.models.runtime_guard import (
    LiveRuntimeError,
    live_demo_locked,
    verify_live_llama_cpp_runtime,
)
from src.rag.live import (
    ARCHITECTURE_LABELS,
    FRESH_KB_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION,
    INSUFFICIENT_EVIDENCE_QUESTION_ID,
    LIVE_ARCHITECTURES,
    LIVE_FAILURE_DECISIONS,
    format_confidence_display,
    format_optional,
    format_threshold_display,
    load_frozen_questions,
    resolve_displayed_confidence,
    resolve_live_locked_threshold,
    run_live_comparison,
    uq_ui_confidence_overlay,
)
from src.rag.schema import ARCHITECTURE_MULTI_AGENT_UQ, RAGCaseResult
from src.retrieval.index import COLLECTION_NAME
from src.retrieval.preflight import IndexPreflightError, validate_index_preflight
from src.utils import create_run_id

def main() -> None:
    st.set_page_config(page_title="V2 RAG Artefact", layout="wide")
    apply_pending_app_page(st.session_state)
    apply_catalogue_prefill_to_live_input(st.session_state)
    with st.sidebar:
        st.radio(
            "Navigate",
            options=["Live RAG Demo", "Benchmark Results", "Benchmark Questions"],
            key="app_page",
        )
    page = st.session_state.get("app_page") or "Live RAG Demo"
    if page == "Benchmark Questions":
        render_benchmark_questions_page()
        return
    if page == "Benchmark Results":
        render_benchmark_results_page()
        return
    render_live_rag_demo()


### Viva explanation

Benchmark Results is read-only frozen tables. Live RAG Demo runs the pipelines.

### Likely viva question

Where are the Streamlit pages defined?

### Answer

`main()` in `V2/app/streamlit_app.py`.

## PHASE 21 — Canonical final live-demo launcher

**This master notebook does not copy `colab_phase21_final_live_demo.ipynb` in full.**

| Field | Value |
|---|---|
| Path | `V2/notebooks/colab_phase21_final_live_demo.ipynb` |
| Role | Viva launch vehicle for the **existing** Streamlit app |
| What it launches | `app/streamlit_app.py` on Colab port 8501 + `proxyPort` |
| What it does not do | Rerun 420 / calibration / judge / statistics / `build_index.py` |

Confirmed local command from `V2/README.md` and `V2/docs/phase21_final_live_demo.md`:

```bash
cd V2
PYTHONPATH=. streamlit run app/streamlit_app.py
```

Colab (from the Phase 21 notebook / docs):

```bash
PYTHONPATH=. python -m streamlit run app/streamlit_app.py \
  --server.port=8501 --server.address=0.0.0.0 --server.headless=true
```

Environment used there: `V2_LIVE_BACKEND=llama_cpp`, `V2_FORBID_MOCK=1`, `V2_REQUIRE_CUDA=1`.


### MASTER CODE ID: M061

**Source file:** `V2/notebooks/colab_phase21_final_live_demo.ipynb`
**Source notebook:** `V2/notebooks/colab_phase21_final_live_demo.ipynb`
**Function/Class:** `Phase 21 live launch (existing app only)`

### What this cell does

Starts the already-implemented Streamlit artefact if the index exists. Does not rebuild research artefacts.

### Libraries used

streamlit, subprocess.

### Inputs

Existing `app/streamlit_app.py`, Phase 6 index, optional GGUF.

### Outputs

Local URL `http://127.0.0.1:8501` or Colab proxy URL.

### Why this matters

The only expensive-looking step allowed in viva is launching the UI, not regenerating 420 cases.

In [ ]:
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

def _port_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0

missing = []
if not (Path("app") / "streamlit_app.py").is_file():
    missing.append("app/streamlit_app.py")
if not (Path("knowledge_base") / "index" / "index_manifest.json").is_file():
    missing.append("knowledge_base/index (restore existing Phase 6 index; do not rebuild)")

if not RUN_LIVE_DEMO:
    print("RUN_LIVE_DEMO is False — not launching.")
elif missing:
    print("LIVE DEMO NOT STARTED. Missing:")
    for item in missing:
        print(" -", item)
    print("Nothing was rebuilt.")
else:
    port = 8501
    on_colab = Path("/content").exists()
    if _port_free(port):
        cmd = [sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py",
               "--server.port", str(port), "--server.headless", "true"]
        if on_colab:
            cmd.extend(["--server.address", "0.0.0.0"])
        env = os.environ.copy()
        env["PYTHONPATH"] = str(Path.cwd()) + os.pathsep + env.get("PYTHONPATH", "")
        if on_colab:
            env["V2_LIVE_BACKEND"] = "llama_cpp"
            env["V2_FORBID_MOCK"] = "1"
        log_path = Path("results/logs/viva_master_streamlit.log")
        log_path.parent.mkdir(parents=True, exist_ok=True)
        proc = subprocess.Popen(cmd, cwd=str(Path.cwd()), env=env,
                                stdout=log_path.open("a"), stderr=subprocess.STDOUT)
        print("Launched Streamlit pid", proc.pid)
        time.sleep(2)
    else:
        print("Port 8501 already in use — assuming Streamlit is running.")
    if Path("/content").exists():
        try:
            from google.colab.output import eval_js
            print("Colab proxy:", eval_js("google.colab.kernel.proxyPort(8501)"))
        except Exception as exc:
            print("Colab proxy not created:", exc)
    else:
        print("Open: http://127.0.0.1:8501")


### Viva explanation

I open the existing app. If the Chroma index or GGUF is missing I say so; I do not silently rebuild.

### Likely viva question

How do you demonstrate the artefact?

### Answer

Launch `app/streamlit_app.py`. Canonical Colab notebook is `colab_phase21_final_live_demo.ipynb`.

## Authoritative result locations

Verified in this checkout as paths (large JSONL is not embedded):

| Artefact | Path |
|---|---|
| Benchmark raw | `V2/results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/` |
| CPU evaluation | `V2/results/processed/phase16_cases.jsonl` |
| CPU summary | `V2/results/metrics/phase16_summary.csv` |
| Judge raw | `V2/results/raw/phase16_judge/phase16_judge_20260828T152623Z_06661255/` |
| Statistics | `V2/results/metrics/phase17_tests.csv` |
| Effect sizes | `V2/results/metrics/phase17_effect_sizes.csv` |
| Assumptions | `V2/results/metrics/phase17_assumptions.csv` |
| Stats summary | `V2/results/metrics/phase17_summary.md` |
| Error cases | `V2/results/analysis/phase18_error_cases.csv` |
| Error summary | `V2/results/analysis/phase18_error_summary.csv` |
| Lock | `V2/results/config/threshold.lock.json` |


In [ ]:
from pathlib import Path
paths = [
    "results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4",
    "results/processed/phase16_cases.jsonl",
    "results/metrics/phase16_summary.csv",
    "results/raw/phase16_judge/phase16_judge_20260828T152623Z_06661255",
    "results/metrics/phase17_tests.csv",
    "results/metrics/phase17_effect_sizes.csv",
    "results/metrics/phase17_assumptions.csv",
    "results/metrics/phase17_summary.md",
    "results/analysis/phase18_error_cases.csv",
    "results/analysis/phase18_error_summary.csv",
    "results/config/threshold.lock.json",
    "data/final/selected_140_questions.csv",
    "data/calibration/calibration_questions.csv",
    "notebooks/colab_phase21_final_live_demo.ipynb",
]
print("Path existence (no file contents dumped):")
for rel in paths:
    p = Path(rel)
    print(("OK     " if p.exists() else "MISSING"), rel)


## Implementation locator

| Professor question | Source file | Function/Class | Master code ID | Short viva answer |
|---|---|---|---|---|
| Where is dataset loading? | `src/data/profile_finqa.py` | `load_finqa` | M004 | `load_dataset("G4KMU/t2-ragbench", "FinQA")` |
| Where are documents collected? | `src/retrieval/pdf_fetch.py` | `collect_corpus_targets` | M013 | TEST+DEV PDFs + 50 train distractors |
| Where is PDF extraction? | `src/retrieval/extract.py` | `extract_pdf_pages` | M015 | PyMuPDF `fitz` |
| Where is chunking? | `src/retrieval/chunking.py` | `split_text` / `chunk_pages` | M016 | 900 / overlap 150 |
| Where are embeddings created? | `src/retrieval/embeddings.py` | `embed_texts` | M017 | `BAAI/bge-small-en-v1.5` |
| Where is ChromaDB? | `src/retrieval/index.py` | `_chroma_client` / `build_knowledge_base` | M018–M019 | collection `finqa_source_pdfs`, cosine |
| Where is retrieval? | `src/retrieval/retriever.py` | `retrieve` | M020 | query embeddings, top_k=4 |
| Where is top-k? | `config/experiment.yaml` + `retriever.py` | `retrieval.top_k` | M020 | 4 |
| Where is Single-Agent RAG? | `src/rag/single_agent.py` | `run_single_agent` | M027 | retrieve → generate, always ANSWER |
| Where is Multi-Agent RAG? | `src/rag/multi_agent.py` | `run_multi_agent` | M032 | retrieve → draft → verify |
| Where is verification? | `src/rag/verification.py` | `compute_verification_result` | M031 | lexical + LLM score, mean, threshold 0.50 |
| Where is uncertainty? | `src/rag/uncertainty.py` | `compute_combined_confidence` | M034 | mean(retrieval, verification) |
| Where is confidence? | `src/rag/uncertainty.py` | `compute_combined_confidence` | M034 | operational decision score |
| Where is the abstention gate? | `src/rag/uncertainty.py` | `apply_abstention_decision` | M034 | ≥ T ANSWER else ABSTAIN |
| Where is Multi-Agent + UQ? | `src/rag/multi_agent_uq.py` | `run_multi_agent_uq` | M035 | verify then confidence gate |
| Where is threshold calibration? | `src/calibration/select.py` | `select_threshold` | M043 | DEV 40, coverage ≥ 0.50 |
| Where is the lock? | `src/calibration/lock.py` | `load_official_lock` | M044 | T=0.65, not TEST |
| Where are prompts? | `src/rag/prompts.py`, `config/prompts.yaml` | `build_*_prompt` | M026 / M029 | evidence + question templates |
| Where is Qwen loaded? | `src/models/llama_cpp_backend.py` | `LlamaCppBackend` | M022 | Q4_K_M GGUF, n_ctx 4096 |
| Where is benchmark execution? | `src/run/benchmark.py` | `run_benchmark` | M048 | 9-case or 420 with `allow_full` |
| Where is evaluation? | `src/evaluation/metrics.py` | `score_case` | M052 | numeric_match CPU |
| Where is the judge? | `src/evaluation/judge.py` | `judge_one_case` | M053 | custom/RAGAS-inspired, no gold |
| Where are statistical tests? | `src/statistics/tests.py` | `mcnemar_exact` | M055 | paired n=140 |
| Where is error analysis? | `src/error_analysis/taxonomy.py` | `assign_category` | M057 | rule-based, not hallucination oracle |
| Where is Streamlit? | `app/streamlit_app.py` | `main` / `render_architecture` | M039 / M060 | three pages |
| Where is the final live demo? | `notebooks/colab_phase21_final_live_demo.ipynb` | launcher | M061 | launches existing app |
| Did you use LangGraph? | — | — | — | No matches for `langgraph` / `StateGraph` under V2 |


## Library / technology map

Built from the copied implementation (not a generic stack list). **LangGraph is not used.** **Plotly is not used** (Phase 17 figures use matplotlib).

| Library / technology | Actual file | Function/Class | Master code ID | What it does here |
|---|---|---|---|---|
| Python | whole V2 package | — | — | implementation language |
| PyYAML | `src/config/loader.py` | `load_experiment_config` | M001 | experiment + prompt YAML |
| Hugging Face datasets | `src/data/profile_finqa.py` | `load_finqa` | M004 | load FinQA splits |
| huggingface_hub | `src/retrieval/pdf_fetch.py` | `download_pdfs` | M014 | fetch page PDFs |
| PyMuPDF (`fitz`) | `src/retrieval/extract.py` | `extract_pdf_pages` | M015 | PDF text |
| sentence-transformers / BGE | `src/retrieval/embeddings.py` | `embed_texts` | M017 | chunk + query vectors |
| ChromaDB | `src/retrieval/index.py`, `retriever.py` | `_chroma_client`, `retrieve` | M018, M020 | persist + query cosine index |
| llama_cpp / llama.cpp | `src/models/llama_cpp_backend.py` | `LlamaCppBackend.generate` | M022 | Qwen3-8B Q4_K_M on CUDA |
| NumPy | `src/statistics/tests.py` | `mcnemar_exact` | M055 | paired arrays |
| SciPy | `src/statistics/tests.py` | `binomtest` / Wilcoxon etc. | M055 | exact tests |
| matplotlib | `src/statistics/figures.py` | figure renderers | (Phase 17 figures) | dissertation plots from saved tables |
| Streamlit | `app/streamlit_app.py` | `main` | M060 | live artefact |
| pandas | not required in the copied core RAG path | — | — | not claimed as a RAG engine |
| LangGraph | **not present** | — | — | sequential Python runners instead |
| Ollama | `src/models/factory.py` | `OllamaBackend` branch | M023 | local-dev only; forbidden in live demo |
| Official RAGAS library | **not used** | judge is custom | M053 | labelled custom/RAGAS-inspired |


## RAG architecture locator

| Architecture | Source file | Function/Class | Master code ID | What it does | How I explain it |
|---|---|---|---|---|---|
| Single-Agent | `src/rag/single_agent.py` | `run_single_agent` | M027 | retrieve → prompt → Qwen → always ANSWER | Baseline RAG, no checker, no abstention |
| Multi-Agent | `src/rag/multi_agent.py` | `run_multi_agent` | M032 | retrieve → draft → verify | Checker scores support; does not rewrite the draft |
| Verification (shared) | `src/rag/verification.py` | `compute_verification_result` | M031 | lexical + LLM mean, VERIFIED if ≥ 0.50 | Informational on Arch 2; input to UQ on Arch 3 |
| Multi-Agent + UQ | `src/rag/multi_agent_uq.py` | `run_multi_agent_uq` | M035 | Arch 2 steps + confidence gate | Same retrieve/verify, then ANSWER/ABSTAIN at T=0.65 |
| Uncertainty | `src/rag/uncertainty.py` | `compute_combined_confidence` | M034 | (R+V)/2 | Operational score, not a probability |


## Retrieval locator

| Step | File | Function | Master code ID |
|---|---|---|---|
| Document list | `pdf_fetch.py` | `collect_corpus_targets` | M013 |
| PDF download | `pdf_fetch.py` | `download_pdfs` | M014 |
| Extraction | `extract.py` | `extract_pdf_pages` | M015 |
| Chunking | `chunking.py` | `split_text` | M016 |
| Embeddings | `embeddings.py` | `embed_texts` | M017 |
| Chroma init/add | `index.py` | `_chroma_client`, `build_knowledge_base` | M018–M019 |
| Query | `retriever.py` | `retrieve` | M020 |


## Dataset / benchmark locator

| Item | Path / code | Master code ID |
|---|---|---|
| Family | T²-RAGBench FinQA `G4KMU/t2-ragbench` | M004 |
| TEST freeze | `data/final/selected_140_questions.csv` | M009 |
| DEV freeze | `data/calibration/calibration_questions.csv` | M012 |
| Loader | `load_frozen_question_rows` | M010 |
| Official job | `run_full_benchmark.py` / Phase 15 notebook | M049–M050 |
| Cases | 140 × 3 = 420 | M046 |


## Calibration / threshold locator

| Item | Location | Master code ID |
|---|---|---|
| Rule | `select_threshold` | M043 |
| Lock loader | `load_official_lock` | M044 |
| File | `results/config/threshold.lock.json` | — |
| T | 0.65 | — |
| Split | DEV 40; `used_frozen_test_140: false` | M046 |
| UI 0.66 band | display-only in Streamlit UQ panel | not a research T |


## Evaluation / statistics / error analysis locator

| Item | File | Function | Master code ID |
|---|---|---|---|
| Numeric correctness | `evaluation/numeric.py` | `numeric_match` | M051 |
| Case metrics | `evaluation/metrics.py` | `score_case` | M052 |
| Judge | `evaluation/judge.py` | `judge_one_case` | M053 |
| McNemar | `statistics/tests.py` | `mcnemar_exact` | M055 |
| RQ wiring | `statistics/analysis.py` | `analyse` | M056 |
| Errors | `error_analysis/taxonomy.py` | `assign_category` | M057 |


## Streamlit / live demo locator

| Item | File | Function | Master code ID |
|---|---|---|---|
| App entry | `app/streamlit_app.py` | `main` | M060 |
| Live comparison | `src/rag/live.py` | `run_live_comparison` | M037 |
| Evidence/verification UI | `app/streamlit_app.py` | `render_architecture` | M039 |
| Locked T in UI | `src/rag/live.py` | `resolve_live_locked_threshold` | M059 |
| Canonical Colab launcher | `notebooks/colab_phase21_final_live_demo.ipynb` | — | M061 |


## Viva question bank

Answers point at the copied implementation. Do not invent extra motivations.

### Dataset

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| Which dataset? | `profile_finqa.py` | `load_finqa` | M004 | FinQA subset of T²-RAGBench only |
| Train/dev/test sizes? | `experiment.yaml` / profile JSON | — | M004 | 6251 / 883 / 1147 |
| Why 140? | `select_140.py` | `freeze_test_140` | M009 | Frozen TEST size in config; seed 42 |
| Why freeze? | `freeze_test_140` note | `freeze_test_140` | M009 | No result-driven resampling |
| Why DEV 40? | `select_calibration.py` | `freeze_calibration` | M012 | Threshold must not use TEST |

### Document processing / PDF / chunking / embeddings / ChromaDB / retrieval

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| Where collected? | `pdf_fetch.py` | `collect_corpus_targets` | M013 | TEST+DEV PDFs + 50 distractors |
| PDF extraction? | `extract.py` | `extract_pdf_pages` | M015 | PyMuPDF |
| Chunk size/overlap? | `chunking.py` | `split_text` | M016 | 900 / 150 |
| How many chunks? | index manifest / yaml | `build_knowledge_base` | M019 | 1239 after Phase 6 build |
| How many PDFs indexed? | yaml `phase6_docs_indexed` | `build_knowledge_base` | M019 | 230 |
| Embeddings? | `embeddings.py` | `embed_texts` | M017 | BGE-small |
| Where is ChromaDB? | `index.py` | `_chroma_client` / `collection.add` | M018–M019 | Persistent cosine collection `finqa_source_pdfs` |
| Where queried? | `retriever.py` | `collection.query` | M020 | inside `retrieve()` |
| top-k? | `experiment.yaml` + `retrieve` | `top_k` | M020 | 4 |
| Gold context in index? | `build_knowledge_base` note | — | M019 | No |

### Prompting

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| Generation prompt? | `prompts.py` / `prompts.yaml` | `build_baseline_prompt` | M026 | Evidence + question; answer once |
| Numeric / ROI handling? | `prompts.yaml` baseline.system | — | M026 | Distinguishes final value vs change vs ROI |
| Insufficient evidence fallback? | same | — | M026 | Write exactly `Evidence is insufficient.` |
| Verification prompt? | `build_multi_agent_verification_prompt` | — | M029 | Reply with one number 0–1 |

### Single-Agent / Multi-Agent / verification / UQ / calibration

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| SA implementation? | `single_agent.py` | `run_single_agent` | M027 | retrieve → generate, always ANSWER |
| Why always answer? | `RAGCaseResult.decision` | — | M025 | No abstention function on this path |
| MA implementation? | `multi_agent.py` | `run_multi_agent` | M032 | + verification |
| Does checker rewrite? | `verification.py` | `compute_verification_result` | M031 | No, scores only |
| Lexical vs LLM? | `verification.py` | mean of both | M031 | `average([lexical_score, llm_score])` |
| VERIFIED threshold? | yaml `verification_threshold` | 0.5 | M031 | status only on Arch 2 |
| Confidence formula? | `uncertainty.py` | `compute_combined_confidence` | M034 | mean(retrieval, verification) |
| Gate? | `apply_abstention_decision` | — | M034 | ≥ T ANSWER else ABSTAIN |
| Is confidence a probability? | — | — | M034 | No. Operational score. No ECE/Brier in repo |
| T=0.65 from where? | `select.py` + lock | `select_threshold` | M043–M044 | DEV 40, coverage floor 0.50 |
| 0.55? | yaml `smoke_threshold` | — | M035 | smoke/pilot fallback in `_resolve_threshold` |
| 0.66 UI band? | Streamlit overlay | display-only | — | not a research threshold |

### Qwen3-8B / llama.cpp / LangGraph

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| Model? | `llama_cpp_backend.py` / yaml | `LlamaCppBackend` | M022 | Qwen3-8B Q4_K_M GGUF |
| n_ctx / max tokens / temp? | yaml `model` | generate kwargs | M022 | 4096 / 512 / 0.1 (judge uses temp 0, 32 tokens) |
| GPU? | GPU check cell | — | — | Official: Tesla T4, CUDA |
| Ollama? | `factory.py` | ollama branch | M023 | local-dev; live demo forbids it |
| LangGraph? | repo search | — | — | Not used. Sequential Python |

### Evaluation / judge / statistics / error analysis

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| Correctness? | `numeric.py` | `numeric_match` | M051 | vs `program_answer` |
| Coverage / selective accuracy? | `score_case` aggregates | Phase 16 | M052 | UQ coverage 78/140, selective 32/78 in saved tables |
| Unsupported-emitted? | `score_case` | — | M052 | answered and displayed incorrect |
| Official RAGAS? | `judge.py` | `METRIC_LABEL` | M053 | No, custom/RAGAS-inspired |
| Gold in judge prompt? | `prompt_contains_forbidden` | — | M053 | Forbidden; leakage check |
| RQ1 p-value? | `mcnemar_exact` | SA vs MA | M055 | 0.6776, not significant (saved Phase 17) |
| Error types? | `taxonomy.py` | `assign_category` | M057 | including retrieval_failure vs incorrect_numerical_reasoning |

### Streamlit / reproducibility / limitations

| Question | File | Function | ID | Answer |
|---|---|---|---|---|
| How to run the artefact? | README / Phase 21 | `streamlit run app/streamlit_app.py` | M061 | from `V2/` with PYTHONPATH=. |
| Live vs lookup? | `live.py` | `run_live_comparison` | M037 | live pipelines, not Phase 15 lookup |
| Pages? | `streamlit_app.py` | `main` | M060 | Live RAG Demo, Benchmark Results, Benchmark Questions |
| Reproducibility? | freezes + lock + SHA pins | Phase 19 | M058 | frozen CSVs, T lock, saved 15–18 |
| Limitation: MA accuracy? | Phase 16/17 tables | — | — | MA displayed 29/140 vs SA 32/140; NS |
| Limitation: confidence? | `uncertainty.py` | — | M034 | not a calibrated probability |
| Limitation: judge? | `judge.py` | — | M053 | custom LLM judge, not official RAGAS |


---
End of master code notebook. Source of truth remains the `.py` files under `V2/`. This file is a viva copy and must stay local.